# RoboMamba 실습 노트북

논문 **RoboMamba: Efficient Vision-Language-Action Model for Robotic Reasoning and Manipulation** (arXiv [2406.04339](https://arxiv.org/abs/2406.04339) v2, NeurIPS 2024) 을 손으로 굴려 보는 실습 7개.

저장소: [lmzpai/roboMamba](https://github.com/lmzpai/roboMamba) · 프로젝트: [robomamba-web](https://sites.google.com/view/robomamba-web)

## 이 노트북을 어디서 돌리나

| | 로컬 주피터 | 구글 코랩 |
|---|---|---|
| 준비물 | Python 3.9+ 와 numpy | 브라우저만 |
| 설치 | 없음 (아래 셀이 확인) | 없음 |
| 실습 1~7 | 전부 동작 | 전부 동작 |
| 부록 (실제 Mamba 모델) | Apple Silicon은 MPS로 동작 | T4 GPU로 동작 |

**코랩에서 여는 법**: 코랩 → `파일 > 노트북 업로드` 에 이 `.ipynb`를 끌어다 놓으면 된다. 아래 첫 셀이 필요한 파일을 스스로 만들기 때문에 저장소를 클론하지 않아도 된다.

**순서**: 위에서 아래로 실행하면 된다. 실습 7만 네트워크가 필요하다 (GitHub에서 소스 6개, 약 200KB).

---


## 0. 부트스트랩 — 이 셀을 먼저 실행

환경을 확인하고, 실습 공용 도구 `tinyssm.py`를 현재 디렉터리에 만든다. 로컬에서 저장소 안에 있으면 기존 파일을 그대로 쓴다.


In [ ]:
import os, sys, platform

print("Python     :", sys.version.split()[0])
print("플랫폼      :", platform.platform())
IN_COLAB = "google.colab" in sys.modules
print("실행 환경   :", "구글 코랩" if IN_COLAB else "로컬 (주피터/IPython)")

try:
    import numpy as np
    print("numpy      :", np.__version__)
except ImportError:
    print("numpy 가 없다 →  %pip install numpy  를 실행하고 이 셀을 다시 돌려라")
    raise

os.makedirs("out", exist_ok=True)
print("\n실습 1~7은 numpy만 쓴다. 추가 설치는 필요 없다.")


### 0-1. 공용 도구 `tinyssm.py` 준비

저장소 `tinyssm.py`와 **같은 내용**이다. 이미 같은 폴더에 있으면 이 셀은 건너뛴다(로컬에서 저장소를 클론한 경우).


In [ ]:
# tinyssm.py 원본 (저장소 lab/tinyssm.py와 동일) — 내용을 읽고 싶으면 펼쳐 보라
_TINYSSM_SOURCE = r'''"""tinyssm — RoboMamba 실습용 공용 도구 (numpy만 사용)

RoboMamba 논문(arXiv 2406.04339v2)과 저장소 lmzpai/roboMamba(확인 2026-09-11)의
Mamba 블록을 numpy로 그대로 옮긴 축소판이다. 설계 원칙 세 개.

1. 저장소와 같은 식·같은 이름. selective_scan()은 modeling_mamba.py의
   selective_scan_ref()를, mamba_block()은 MyMamba.forward()를 따른다.
2. GPU·PyTorch 없이 1초 안에 돈다. 차원만 작게 줄였다.
3. 학습이 필요한 실습은 수동 역전파 + Adam으로 처리한다.

참고: 실제 하이퍼파라미터 (state-spaces/mamba-2.8b-hf config.json, 확인 2026-09-11)
  d_model=2560, n_layer=64, d_state(N)=16, d_conv=4, expand=2,
  d_inner=5120, dt_rank(time_step_rank)=160, vocab=50280
"""

import math
import time

import numpy as np

# ─────────────────────────────────────────────────────────────────────────────
# 활성 함수
# ─────────────────────────────────────────────────────────────────────────────


def softplus(x):
    """log(1+exp(x)). Δ를 항상 양수로 만드는 데 쓴다 (delta_softplus=True)."""
    return np.log1p(np.exp(-np.abs(x))) + np.maximum(x, 0.0)


def silu(x):
    """x * sigmoid(x). Mamba 블록의 활성 함수 (config의 hidden_act='silu')."""
    return x / (1.0 + np.exp(-x))


def relu(x):
    return np.maximum(x, 0.0)


def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)


# ─────────────────────────────────────────────────────────────────────────────
# 선택적 스캔 (SSM의 심장)
# ─────────────────────────────────────────────────────────────────────────────


def selective_scan(u, delta, A, B, C, D=None, z=None, return_states=False):
    """저장소 modeling_mamba.py의 selective_scan_ref()를 numpy로 옮긴 것.

    논문 식 (2)(3)(4)에 해당한다.
        Ā = exp(ΔA)                    ... 식 (2)
        B̄ ≈ Δ·B                        ... 식 (3)의 1차 근사(저장소와 동일)
        h_t = Ā h_{t-1} + B̄ x_t        ... 식 (4)
        y_t = C h_t

    인자
        u     (L, Dm)  입력 시퀀스 (저장소의 conv+silu 통과 후 x)
        delta (L, Dm)  시각 간격 Δ. 토큰마다·채널마다 다르다 → 이것이 '선택성'
        A     (Dm, N)  상태 행렬. 저장소는 A = -exp(A_log)이라 항상 음수
        B     (L, N)   입력 행렬. 토큰마다 다르다 (S6의 핵심)
        C     (L, N)   출력 행렬. 토큰마다 다르다
        D     (Dm,)    잔차(skip) 항
        z     (L, Dm)  게이트. out = out * silu(z)

    반환
        y (L, Dm), 마지막 상태 h (Dm, N) [, 모든 상태 (L, Dm, N)]

    주의: 저장소는 einsum('bdl,dn->bdln')으로 배치를 함께 처리하지만
    여기서는 배치 1개만 다룬다. 수식은 완전히 같다.
    """
    L, Dm = u.shape
    N = A.shape[1]
    h = np.zeros((Dm, N), dtype=np.float64)
    y = np.zeros((L, Dm), dtype=np.float64)
    states = np.zeros((L, Dm, N), dtype=np.float64) if return_states else None

    for t in range(L):
        dA = np.exp(delta[t][:, None] * A)                     # (Dm, N)  식 (2)
        dBu = delta[t][:, None] * B[t][None, :] * u[t][:, None]  # (Dm, N)  식 (3)
        h = dA * h + dBu                                        # 식 (4) 앞부분
        y[t] = (h * C[t][None, :]).sum(axis=-1)                 # 식 (4) 뒷부분
        if return_states:
            states[t] = h

    if D is not None:
        y = y + u * D[None, :]
    if z is not None:
        y = y * silu(z)
    return (y, h, states) if return_states else (y, h)


def causal_depthwise_conv1d(x, w, b=None):
    """저장소 conv1d(groups=d_inner, kernel_size=4, padding=3)의 인과 버전.

    채널별로 독립인(depthwise) 1D 합성곱. 왼쪽만 0으로 채워 미래를 보지 않는다.
    Mamba가 '직전 3토큰'이라는 짧은 국소 문맥을 보는 유일한 장치다.
    """
    L, Dm = x.shape
    k = w.shape[1]
    pad = np.zeros((k - 1, Dm), dtype=x.dtype)
    xp = np.concatenate([pad, x], axis=0)
    out = np.zeros_like(x)
    for t in range(L):
        win = xp[t:t + k]                       # (k, Dm)
        out[t] = (win * w.T).sum(axis=0)
    if b is not None:
        out = out + b[None, :]
    return out


# ─────────────────────────────────────────────────────────────────────────────
# Mamba 블록
# ─────────────────────────────────────────────────────────────────────────────


def mamba_params(d_model, d_state=16, d_conv=4, expand=2, dt_rank=None, seed=0):
    """MyMamba.__init__()과 같은 모양·같은 초기화의 가중치 묶음을 만든다."""
    rng = np.random.default_rng(seed)
    d_inner = expand * d_model
    if dt_rank is None:
        dt_rank = math.ceil(d_model / 16)       # 저장소의 dt_rank='auto'

    # dt_proj 초기화: softplus(bias)가 [dt_min, dt_max]에 들어오게 (저장소와 동일)
    dt_min, dt_max, dt_floor = 0.001, 0.1, 1e-4
    dt = np.exp(rng.random(d_inner) * (math.log(dt_max) - math.log(dt_min))
                + math.log(dt_min)).clip(min=dt_floor)
    inv_dt = dt + np.log(-np.expm1(-dt))        # softplus의 역함수
    dt_std = dt_rank ** -0.5

    return {
        "d_model": d_model, "d_inner": d_inner, "d_state": d_state,
        "d_conv": d_conv, "dt_rank": dt_rank,
        "in_proj": rng.normal(0, d_model ** -0.5, (d_model, 2 * d_inner)),
        "conv_w": rng.normal(0, 0.5, (d_inner, d_conv)),
        "conv_b": np.zeros(d_inner),
        "x_proj": rng.normal(0, d_inner ** -0.5, (d_inner, dt_rank + 2 * d_state)),
        "dt_proj_w": rng.uniform(-dt_std, dt_std, (dt_rank, d_inner)),
        "dt_proj_b": inv_dt,
        # S4D-real 초기화: A_log = log(1..N) → A = -exp(A_log) = -(1..N)
        "A_log": np.log(np.tile(np.arange(1, d_state + 1, dtype=np.float64),
                                (d_inner, 1))),
        "D": np.ones(d_inner),
        "out_proj": rng.normal(0, d_inner ** -0.5, (d_inner, d_model)),
    }


def mamba_block(x, W, return_states=False):
    """MyMamba.forward()의 numpy 축소판. x: (L, d_model) → (L, d_model)

    순서: in_proj로 x와 z로 갈라짐 → 인과 conv → SiLU → x_proj로 Δ·B·C를
    뽑음 → dt_proj+softplus로 Δ → selective_scan → out_proj.
    """
    N, dt_rank = W["d_state"], W["dt_rank"]
    xz = x @ W["in_proj"]                                   # (L, 2*d_inner)
    xin, z = np.split(xz, 2, axis=-1)
    xin = silu(causal_depthwise_conv1d(xin, W["conv_w"], W["conv_b"]))
    dbc = xin @ W["x_proj"]                                 # (L, dt_rank+2N)
    dt, Bm, Cm = np.split(dbc, [dt_rank, dt_rank + N], axis=-1)
    delta = softplus(dt @ W["dt_proj_w"] + W["dt_proj_b"])  # (L, d_inner)
    A = -np.exp(W["A_log"])                                 # 항상 음수 → 감쇠
    out = selective_scan(xin, delta, A, Bm, Cm, D=W["D"], z=z,
                         return_states=return_states)
    y = out[0]
    return (y @ W["out_proj"],) + out[1:]


# ─────────────────────────────────────────────────────────────────────────────
# 비교 대상: 어텐션 (O(L²))
# ─────────────────────────────────────────────────────────────────────────────


def causal_attention(x, Wq, Wk, Wv, Wo):
    """디코더 전용 트랜스포머 블록의 어텐션. 계산량이 L²에 비례한다."""
    q, k, v = x @ Wq, x @ Wk, x @ Wv
    scores = q @ k.T / math.sqrt(q.shape[-1])               # (L, L) ← 여기가 L²
    mask = np.triu(np.ones_like(scores), k=1) * -1e9
    return (softmax(scores + mask) @ v) @ Wo


# ─────────────────────────────────────────────────────────────────────────────
# 6D 회전 표현 (저장소 manip.py의 loss_6d_rot)
# ─────────────────────────────────────────────────────────────────────────────


def gram_schmidt_6d(d6):
    """저장소 manip.py의 bgs(). 6개 숫자 → 정규직교 회전행렬 3×3.

    d6 (..., 6)을 두 벡터 a1, a2로 보고
      b1 = normalize(a1)
      b2 = normalize(a2 - (b1·a2) b1)
      b3 = b1 × b2
    결과 [b1 b2 b3]는 항상 SO(3) 원소다. 헤드가 9개 숫자를 그냥 뱉으면
    직교성이 깨지는데, 6D 표현은 어떤 6개 숫자가 나와도 유효한 회전이 된다.
    """
    d6 = np.asarray(d6, dtype=np.float64).reshape(-1, 2, 3)
    a1, a2 = d6[:, 0, :], d6[:, 1, :]
    b1 = a1 / (np.linalg.norm(a1, axis=1, keepdims=True) + 1e-12)
    a2p = a2 - (b1 * a2).sum(axis=1, keepdims=True) * b1
    b2 = a2p / (np.linalg.norm(a2p, axis=1, keepdims=True) + 1e-12)
    b3 = np.cross(b1, b2)
    return np.stack([b1, b2, b3], axis=-1)                  # (M, 3, 3) 열이 축


def geodesic_loss(R_pred, R_gt):
    """논문 식 (6). arccos((tr(Rgt^T Rpred) − 1) / 2) = 두 회전 사이 각도(라디안).

    저장소 manip.py의 bgdR()과 같다. clamp로 arccos 정의역을 지킨다.
    """
    Rd = np.einsum("mij,mik->mjk", R_gt, R_pred)            # Rgt^T @ Rpred
    tr = np.trace(Rd, axis1=1, axis2=2)
    return np.arccos(np.clip(0.5 * (tr - 1.0), -1 + 1e-6, 1 - 1e-6))


# ─────────────────────────────────────────────────────────────────────────────
# 수동 역전파 MLP + Adam (학습이 필요한 실습용)
# ─────────────────────────────────────────────────────────────────────────────


class Linear:
    def __init__(self, inp, oup, rng, bias=True, xavier=True):
        # 저장소 manip.py는 정책 헤드를 xavier_uniform_ + bias=0으로 초기화한다
        if xavier:
            lim = math.sqrt(6.0 / (inp + oup))
            self.W = rng.uniform(-lim, lim, (inp, oup))
        else:
            self.W = rng.normal(0, inp ** -0.5, (inp, oup))
        self.b = np.zeros(oup) if bias else None
        self.gW = np.zeros_like(self.W)
        self.gb = np.zeros_like(self.b) if bias is not None and self.b is not None else None

    def __call__(self, x):
        self.x = x
        out = x @ self.W
        return out if self.b is None else out + self.b

    def backward(self, g):
        self.gW[...] = self.x.T @ g
        if self.b is not None:
            self.gb[...] = g.sum(axis=0)
        return g @ self.W.T

    def params(self):
        return [(self.W, self.gW)] + ([(self.b, self.gb)] if self.b is not None else [])

    def n_params(self):
        return self.W.size + (self.b.size if self.b is not None else 0)


class SpecialMLP:
    """저장소 manip.py의 SpecialMLP을 그대로 옮긴 것.

        fc1: Linear(inp, inp//2)   → ReLU
        fc2: Linear(inp//2, inp//4)→ ReLU
        fc3: Linear(inp//4, oup, bias=False)

    RoboMamba의 정책 헤드는 이 MLP 두 개다(head_type='two_mlp').
    head1: oup=2 (접촉점 x, y) / head2: oup=6 (6D 회전)
    """

    def __init__(self, inp, oup, rng):
        self.fc1 = Linear(inp, inp // 2, rng)
        self.fc2 = Linear(inp // 2, inp // 4, rng)
        self.fc3 = Linear(inp // 4, oup, rng, bias=False)

    def __call__(self, x):
        self.h1 = relu(self.fc1(x))
        self.h2 = relu(self.fc2(self.h1))
        return self.fc3(self.h2)

    def backward(self, g):
        g = self.fc3.backward(g)
        g = self.fc2.backward(g * (self.h2 > 0))
        return self.fc1.backward(g * (self.h1 > 0))

    def params(self):
        return self.fc1.params() + self.fc2.params() + self.fc3.params()

    def n_params(self):
        return self.fc1.n_params() + self.fc2.n_params() + self.fc3.n_params()


class Adam:
    """저장소가 쓰는 AdamW의 weight-decay 없는 버전 (논문 4.1절: AdamW, β=(0.9,0.999))."""

    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        self.p = list(params)
        self.lr, self.b1, self.b2, self.eps = lr, betas[0], betas[1], eps
        self.m = [np.zeros_like(w) for w, _ in self.p]
        self.v = [np.zeros_like(w) for w, _ in self.p]
        self.t = 0

    def step(self):
        self.t += 1
        for i, (w, g) in enumerate(self.p):
            self.m[i] = self.b1 * self.m[i] + (1 - self.b1) * g
            self.v[i] = self.b2 * self.v[i] + (1 - self.b2) * g * g
            mh = self.m[i] / (1 - self.b1 ** self.t)
            vh = self.v[i] / (1 - self.b2 ** self.t)
            w -= self.lr * mh / (np.sqrt(vh) + self.eps)

    def zero_grad(self):
        for _, g in self.p:
            g[...] = 0.0


# ─────────────────────────────────────────────────────────────────────────────
# 측정·출력 도구
# ─────────────────────────────────────────────────────────────────────────────


def timeit(fn, repeat=3):
    """가장 빠른 실행 시간(ms). 배경 잡음을 줄이려고 최소값을 쓴다."""
    best = float("inf")
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn()
        best = min(best, time.perf_counter() - t0)
    return best * 1000.0


def fmt_params(n):
    if n >= 1e9:
        return f"{n / 1e9:.2f}B"
    if n >= 1e6:
        return f"{n / 1e6:.2f}M"
    if n >= 1e3:
        return f"{n / 1e3:.1f}K"
    return str(n)


def table(headers, rows, highlight=None):
    """터미널용 표. highlight는 강조할 행 인덱스 집합."""
    cols = [max(len(str(h)), *(len(str(r[i])) for r in rows))
            for i, h in enumerate(headers)]
    line = "─".join("─" * c for c in cols)
    out = [" │ ".join(str(h).ljust(c) for h, c in zip(headers, cols)), line]
    for j, r in enumerate(rows):
        mark = " ◀" if highlight and j in highlight else ""
        out.append(" │ ".join(str(v).ljust(c) for v, c in zip(r, cols)) + mark)
    return "\n".join(out)


def save_svg(path, width, height, body, title=""):
    """의존성 없이 그림을 남긴다. 브라우저로 열어 본다."""
    svg = (
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" '
        f'viewBox="0 0 {width} {height}" font-family="ui-sans-serif,system-ui,sans-serif">'
        f'<rect width="{width}" height="{height}" fill="#fbfbf9"/>'
        + (f'<text x="{width/2}" y="24" text-anchor="middle" font-size="15" '
           f'font-weight="600" fill="#1a1a18">{title}</text>' if title else "")
        + body + "</svg>"
    )
    with open(path, "w") as f:
        f.write(svg)
    return path


def polyline(pts, color="#2f6f4f", w=2, dash=None):
    d = f' stroke-dasharray="{dash}"' if dash else ""
    p = " ".join(f"{x:.1f},{y:.1f}" for x, y in pts)
    return f'<polyline points="{p}" fill="none" stroke="{color}" stroke-width="{w}"{d}/>'


def axes(x0, y0, x1, y1, xlabel="", ylabel=""):
    s = (f'<line x1="{x0}" y1="{y1}" x2="{x1}" y2="{y1}" stroke="#8a8a82"/>'
         f'<line x1="{x0}" y1="{y0}" x2="{x0}" y2="{y1}" stroke="#8a8a82"/>')
    if xlabel:
        s += (f'<text x="{(x0+x1)/2}" y="{y1+34}" text-anchor="middle" '
              f'font-size="11" fill="#55554f">{xlabel}</text>')
    if ylabel:
        s += (f'<text x="{x0-38}" y="{(y0+y1)/2}" text-anchor="middle" font-size="11" '
              f'fill="#55554f" transform="rotate(-90 {x0-38} {(y0+y1)/2})">{ylabel}</text>')
    return s


PAPER = "RoboMamba (arXiv 2406.04339v2, NeurIPS 2024)"
REPO = "lmzpai/roboMamba (main, 확인 2026-09-11)"


def header(n, title, guide):
    bar = "═" * 74
    return (f"\n{bar}\n  실습 {n} · {title}\n  가이드 {guide} · {PAPER}\n{bar}")
'''
print(f'tinyssm 원본 {len(_TINYSSM_SOURCE):,} 바이트 준비')


In [ ]:
import os
if os.path.exists("tinyssm.py"):
    print("tinyssm.py 가 이미 있다 — 그것을 쓴다")
else:
    src = _TINYSSM_SOURCE
    open("tinyssm.py", "w", encoding="utf-8").write(src)
    print(f"tinyssm.py 생성 ({len(src):,} 바이트)")
import tinyssm as T
print("import OK ·", T.PAPER)


---

## 실습 1 · 상태공간모델을 손으로 굴려 보기

```
논문 3.1절의 식 (1)~(4)가 실제로 무엇을 계산하는지 숫자로 확인한다.
    h'(t) = A h(t) + B x(t);  y(t) = C h(t)      ... 식 (1)  연속
    Ā = exp(ΔA)                                   ... 식 (2)  이산화
    B̄ = (ΔA)^-1 (exp(ΔA) − I) · ΔB               ... 식 (3)  이산화
    h_t = Ā h_{t-1} + B̄ x_t;  y_t = C h_t        ... 식 (4)  순환

실행: python3 lab1_ssm_recurrence.py     (약 1초, out/lab1_state.svg 생성)
```

원본 스크립트: `lab1_ssm_recurrence.py`


In [ ]:
"""실습 1 · 상태공간모델을 손으로 굴려 보기 — 가이드 1.4 · 1.5 · 2.4

논문 3.1절의 식 (1)~(4)가 실제로 무엇을 계산하는지 숫자로 확인한다.
    h'(t) = A h(t) + B x(t);  y(t) = C h(t)      ... 식 (1)  연속
    Ā = exp(ΔA)                                   ... 식 (2)  이산화
    B̄ = (ΔA)^-1 (exp(ΔA) − I) · ΔB               ... 식 (3)  이산화
    h_t = Ā h_{t-1} + B̄ x_t;  y_t = C h_t        ... 식 (4)  순환

실행: python3 lab1_ssm_recurrence.py     (약 1초, out/lab1_state.svg 생성)
"""

import os
import sys

import numpy as np

import tinyssm as T

OUT = os.path.join(".", "out")
os.makedirs(OUT, exist_ok=True)
rng = np.random.default_rng(0)

print(T.header(1, "상태공간모델을 손으로 굴려 보기", "1.4 · 1.5 · 2.4"))

# ─────────────────────────────────────────────────────────────────────────────
# (1) 이산화 — 식 (2)(3)이 하는 일
# ─────────────────────────────────────────────────────────────────────────────
print("""
(1) 이산화: 연속 시스템을 '한 토큰 = 한 스텝'으로 바꾸기
────────────────────────────────────────────────────────────────
식 (1)은 시간이 연속인 미분방정식이다. 토큰은 띄엄띄엄 오니까
'Δ만큼의 시간 동안 x가 일정했다'고 보고(zero-order hold) 적분해 둔다.
그 결과가 식 (2)(3)의 Ā, B̄이고, 식 (4)는 그냥 for 문이다.""")

A_scalar, B_scalar = -1.0, 1.0     # 상태 1개짜리 최소 예제
rows = []
for d in (0.001, 0.01, 0.1, 1.0, 5.0):
    Abar = np.exp(d * A_scalar)                             # 식 (2)
    Bbar_exact = (1.0 / (d * A_scalar)) * (Abar - 1.0) * d * B_scalar  # 식 (3)
    Bbar_code = d * B_scalar                                # 저장소가 쓰는 근사
    rows.append([f"{d:g}", f"{Abar:.4f}", f"{Bbar_exact:.4f}", f"{Bbar_code:.4f}",
                 f"{abs(Bbar_exact - Bbar_code) / abs(Bbar_exact) * 100:5.1f}%"])
print()
print(T.table(["Δ", "Ā=exp(ΔA)", "B̄ 식(3) 정확", "B̄ 저장소 근사(ΔB)", "차이"], rows))
print("""
읽는 법 (A = −1로 고정했을 때)
  · Ā는 '과거를 얼마나 남기나'. Δ가 작으면 1에 가깝고(거의 그대로 보존),
    Δ가 크면 0에 가깝다(과거를 지운다). A가 음수라서 항상 0~1 사이다.
  · B̄는 '지금 입력을 얼마나 받나'. Δ가 크면 크다.
  · 즉 Δ 하나가 "지금 것을 받아들이고 과거를 지운다"(Δ 큼) ↔
    "지금 것을 무시하고 과거를 지킨다"(Δ 작음)를 동시에 조절한다.
  · 저장소는 식 (3)을 정확히 계산하지 않고 B̄ ≈ Δ·B로 쓴다.
    Δ가 작은 영역(0.001~0.1, 실제 학습된 범위)에서 오차가 5% 미만이라
    Mamba 원 논문부터 이 근사를 쓴다. 저장소 확인 위치:
    modeling_mamba.py selective_scan_ref() 중 deltaB_u = einsum('bdl,bnl,bdl->bdln')""")

# ─────────────────────────────────────────────────────────────────────────────
# (2) 상태는 고정 크기 — 여기서 '선형 복잡도'가 나온다
# ─────────────────────────────────────────────────────────────────────────────
print("""
(2) 시퀀스가 길어져도 상태 크기는 그대로
────────────────────────────────────────────────────────────────""")
d_model = 32
W = T.mamba_params(d_model, seed=1)
rows = []
for L in (8, 64, 256, 1024):
    x = rng.normal(0, 1, (L, d_model))
    y, h = T.mamba_block(x, W)
    kv = 2 * L * d_model            # 트랜스포머가 들고 있어야 하는 K·V 캐시 크기
    rows.append([L, f"{y.shape}", f"{h.shape}", h.size, kv, f"{kv / h.size:.1f}×"])
print(T.table(["L(토큰 수)", "출력 모양", "상태 h 모양", "상태 숫자 개수",
               "트랜스포머 KV 캐시", "배"], rows))
print(f"""
상태 h는 (d_inner={W['d_inner']}, N={W['d_state']}) = {W['d_inner'] * W['d_state']}개 숫자로
L과 무관하게 고정이다. 토큰을 하나 더 넣는 비용도 항상 같다 → O(L).
트랜스포머는 토큰이 늘면 KV 캐시가 같이 늘고, 어텐션이 L²에 비례한다.
논문 초록의 "linear inference complexity"가 이 표의 4·5열 '배' 변화다.
(토큰이 아주 적으면 SSM 상태가 오히려 더 크다. 중요한 건 절대 크기가 아니라
 L이 늘 때 한쪽은 그대로, 다른 쪽은 비례해 커진다는 점이다.)
실제 값: d_model=2560 → 상태 5120×16 = 81,920개 (블록당), 64블록 = 524만개 고정.""")

# ─────────────────────────────────────────────────────────────────────────────
# (3) Δ가 '선택'하는 것을 눈으로 — 같은 입력, 다른 Δ
# ─────────────────────────────────────────────────────────────────────────────
print("""
(3) 같은 입력에 Δ만 바꿔 보기 — 기억이 남는 길이가 달라진다
────────────────────────────────────────────────────────────────""")
L, N = 60, 1
u = np.zeros((L, 1))
u[5] = 1.0          # 5번 토큰에만 정보 한 방울
A = np.array([[-1.0]])
Bc = np.ones((L, N))
Cc = np.ones((L, N))

curves, rows = {}, []
for d_val in (0.02, 0.1, 0.5, 2.0):
    delta = np.full((L, 1), d_val)
    y, _, states = T.selective_scan(u, delta, A, Bc, Cc, return_states=True)
    tr = states[:, 0, 0]
    peak = tr.max()
    # 신호가 최고값의 10%까지 줄어드는 데 걸린 토큰 수 = '기억 반경'
    after = tr[5:]
    life = int(np.argmax(after < peak * 0.1)) if (after < peak * 0.1).any() else L - 5
    curves[d_val] = tr
    rows.append([f"{d_val:g}", f"{np.exp(d_val * -1.0):.4f}", f"{peak:.4f}", life])
print(T.table(["Δ", "Ā=exp(ΔA)", "5번 토큰 직후 상태", "10%까지 남는 토큰 수"], rows))
print("""
Δ=0.02면 정보가 100토큰 넘게 살아 있고(장기 기억), Δ=2.0이면 2토큰 만에 사라진다.
Mamba의 S6는 이 Δ를 **입력이 정하게** 만든 것이다(Δ = softplus(dt_proj(x_proj(x)))).
"지금 토큰이 중요하다" → Δ를 키워 상태를 갈아 끼우고,
"관계없는 토큰이다" → Δ를 줄여 지나 보낸다. 논문 3.1절이 말하는
'content-aware reasoning'이 이것이고, 실습 3에서 학습으로 확인한다.""")

# ─────────────────────────────────────────────────────────────────────────────
# (4) 저장소 대응
# ─────────────────────────────────────────────────────────────────────────────
print(f"""
(4) 저장소 대응 — {T.REPO}
────────────────────────────────────────────────────────────────
  논문 식 (2) Ā=exp(ΔA)  → modeling_mamba.py  deltaA = torch.exp(einsum('bdl,dn->bdln', delta, A))
  논문 식 (3) B̄          → 같은 함수          deltaB_u = einsum('bdl,bnl,bdl->bdln', delta, B, u)
  논문 식 (4) 순환        → 같은 함수          x = deltaA[:,:,i] * x + deltaB_u[:,:,i]
  A가 음수인 근거         → MyMamba.forward()  A = -torch.exp(self.A_log.float())
  A의 초기값 (S4D-real)   → MyMamba.__init__() A_log = log(arange(1, d_state+1)) → A = −(1..16)
  Δ를 입력에서 만드는 곳  → MyMamba.__init__() x_proj(d_inner → dt_rank+2N), dt_proj(dt_rank → d_inner)

해 볼 것
  · A의 초기값을 −1..−16이 아니라 전부 −1로 바꾸면 (3)의 '기억 반경'이
    16개 채널에서 다 같아진다. 왜 굳이 1..16으로 흩어 놓았을까?
  · Δ를 (L,1) 상수가 아니라 u에 비례하게 주면 무슨 일이 생기는가?""")

# ─────────────────────────────────────────────────────────────────────────────
# 그림
# ─────────────────────────────────────────────────────────────────────────────
X0, Y0, X1, Y1 = 70, 50, 700, 300
body = T.axes(X0, Y0, X1, Y1, "토큰 위치 t (5번 토큰에만 입력 1.0)", "상태 h_t")
body += (f'<line x1="{X0 + 5 / L * (X1 - X0):.0f}" y1="{Y0}" '
         f'x2="{X0 + 5 / L * (X1 - X0):.0f}" y2="{Y1}" stroke="#c9c9c0" '
         f'stroke-dasharray="3 3"/>')
colors = {0.02: "#2f6f4f", 0.1: "#3f6fa8", 0.5: "#b8863f", 2.0: "#a8443f"}
ymax = max(c.max() for c in curves.values()) * 1.1
for i, (d_val, tr) in enumerate(curves.items()):
    pts = [(X0 + t / L * (X1 - X0), Y1 - tr[t] / ymax * (Y1 - Y0)) for t in range(L)]
    body += T.polyline(pts, colors[d_val], 2)
    body += (f'<text x="{X1 + 6}" y="{Y1 - tr[6] / ymax * (Y1 - Y0) + 4}" font-size="11" '
             f'fill="{colors[d_val]}">Δ={d_val:g}</text>')
p = T.save_svg(os.path.join(OUT, "lab1_state.svg"), 800, 340, body,
               "Δ 하나가 기억의 길이를 정한다 — h_t = exp(ΔA)·h_{t−1} + ΔB·x_t")
print(f"\n그림 저장: {p}")


In [ ]:
from IPython.display import SVG, display
display(SVG(filename="out/lab1_state.svg"))


---

## 실습 2 · 어텐션은 왜 느리고 SSM은 왜 빠른가

```
논문의 속도 주장(그림 1: RoboMamba 9.0 Hz vs OpenVLA 3.4 Hz vs ManipLLM 1.1 Hz,
4.2절: "7 times faster than LLaMA-AdapterV2")이 어디서 오는지 직접 재 본다.

실행: python3 lab2_linear_vs_quadratic.py   (약 15초, out/lab2_scaling.svg 생성)
```

원본 스크립트: `lab2_linear_vs_quadratic.py`


In [ ]:
"""실습 2 · 어텐션은 왜 느리고 SSM은 왜 빠른가 — 가이드 1.3 · 1.5 · 2.6

논문의 속도 주장(그림 1: RoboMamba 9.0 Hz vs OpenVLA 3.4 Hz vs ManipLLM 1.1 Hz,
4.2절: "7 times faster than LLaMA-AdapterV2")이 어디서 오는지 직접 재 본다.

실행: python3 lab2_linear_vs_quadratic.py   (약 15초, out/lab2_scaling.svg 생성)
"""

import math
import os
import sys

import numpy as np

import tinyssm as T

OUT = os.path.join(".", "out")
os.makedirs(OUT, exist_ok=True)
rng = np.random.default_rng(0)

print(T.header(2, "어텐션은 왜 느리고 SSM은 왜 빠른가", "1.3 · 1.5 · 2.6"))

# ─────────────────────────────────────────────────────────────────────────────
# (1) 토큰 수를 두 배씩 늘리며 한 블록의 시간을 잰다
# ─────────────────────────────────────────────────────────────────────────────
D = 64
Wq, Wk, Wv, Wo = (rng.normal(0, D ** -0.5, (D, D)) for _ in range(4))
Wm = T.mamba_params(D, seed=1)

print("""
(1) 블록 하나를 통과하는 시간 (numpy 단일 스레드, 상대 비교용)
────────────────────────────────────────────────────────────────""")
rows, meas = [], {}
base_att = base_ssm = None
for L in (64, 128, 256, 512, 1024, 2048):
    x = rng.normal(0, 1, (L, D))
    t_att = T.timeit(lambda: T.causal_attention(x, Wq, Wk, Wv, Wo))
    t_ssm = T.timeit(lambda: T.mamba_block(x, Wm))
    if base_att is None:
        base_att, base_ssm = t_att, t_ssm
    meas[L] = (t_att, t_ssm)
    rows.append([L, f"{t_att:8.2f}", f"×{t_att / base_att:5.1f}",
                 f"{t_ssm:8.2f}", f"×{t_ssm / base_ssm:5.1f}",
                 f"{L * L:,}", f"{L:,}"])
print(T.table(["L", "어텐션 ms", "L=64 대비", "SSM ms", "L=64 대비",
               "L² (이론)", "L (이론)"], rows))
print("""
어텐션은 L을 32배 늘리면 시간이 수백 배로 뛴다(L² 항). SSM은 거의 32배다.
여기서 SSM의 절대 시간이 더 큰 것은 numpy for 문 때문이고(실제로는 CUDA
병렬 스캔 커널이 담당), 비교해야 할 것은 '증가율'이다.
로봇에서 이 차이가 중요한 이유: 이미지 한 장이 CLIP ViT-L/14@224에서
패치 토큰 256개를 만든다. 손목 카메라를 더하거나 과거 프레임을 쌓으면
L은 금방 1000을 넘고, 그때 어텐션은 L²로 벌을 받는다.""")

# ─────────────────────────────────────────────────────────────────────────────
# (2) 생성 단계: 자기회귀로 한 토큰씩 뽑을 때
# ─────────────────────────────────────────────────────────────────────────────
print("""
(2) 자기회귀 생성 — 토큰을 하나 더 뽑는 비용
────────────────────────────────────────────────────────────────
로봇 계획("다음 5스텝?")은 문장을 생성한다. 트랜스포머는 KV 캐시를 써도
새 토큰이 이전 t개 토큰 모두와 내적을 해야 하므로 t에 비례해 느려진다.
SSM은 고정 크기 상태 하나만 갱신하므로 언제나 같은 비용이다.""")

d_inner, N = Wm["d_inner"], Wm["d_state"]
rows = []
for ctx in (256, 512, 1024, 2048):
    # 트랜스포머: 새 토큰 q(1,D)와 캐시 K,V(ctx,D)
    Kc, Vc = rng.normal(0, 1, (ctx, D)), rng.normal(0, 1, (ctx, D))
    q = rng.normal(0, 1, (1, D))

    def step_attn():
        s = q @ Kc.T / math.sqrt(D)
        return (T.softmax(s) @ Vc) @ Wo

    # SSM: 고정 크기 상태 (d_inner, N) 하나 갱신
    h = rng.normal(0, 1, (d_inner, N))
    dA = np.exp(rng.normal(0, 0.1, (d_inner, N)))
    dBu = rng.normal(0, 1, (d_inner, N))
    Cc = rng.normal(0, 1, N)

    def step_ssm():
        hh = dA * h + dBu
        return (hh * Cc[None, :]).sum(-1)

    t_a = T.timeit(step_attn, repeat=5)
    t_s = T.timeit(step_ssm, repeat=5)
    rows.append([ctx, f"{ctx * D * 2:,}", f"{t_a * 1000:7.1f}",
                 f"{d_inner * N * 2:,}", f"{t_s * 1000:7.1f}"])
print(T.table(["문맥 길이", "어텐션이 읽는 숫자", "어텐션 µs",
               "SSM이 읽는 숫자", "SSM µs"], rows))
print("""
'읽는 숫자' 열이 핵심이다. 어텐션은 문맥이 길어질수록 매 스텝 더 많은
메모리를 훑고(그래서 실제 GPU에서도 메모리 대역폭에 묶인다), SSM은 항상
d_inner×N = 5120×16 = 81,920개만 본다. 논문 그림 1의 9.0 Hz vs 1.1 Hz는
이 성질 + 7B→2.8B 크기 축소가 함께 만든 결과다.""")

# ─────────────────────────────────────────────────────────────────────────────
# (3) 논문의 속도 수치를 제어 주기로 옮겨 읽기
# ─────────────────────────────────────────────────────────────────────────────
print("""
(3) 논문 그림 1의 Hz를 '한 번 판단에 걸리는 시간'으로
────────────────────────────────────────────────────────────────""")
rows = [
    ["ManipLLM (7B, 트랜스포머)", "1.1", f"{1000 / 1.1:6.0f}", "41.3M (0.5%)", "×1.0"],
    ["OpenVLA (7B, 트랜스포머)", "3.4", f"{1000 / 3.4:6.0f}", "7.0B (100%)", "×3.1"],
    ["RoboMamba (2.8B, Mamba)", "9.0", f"{1000 / 9.0:6.0f}", "3.7M (0.1%)", "×8.2"],
]
print(T.table(["모델", "추론 Hz", "1회 ms", "파인튜닝 파라미터", "ManipLLM 대비"],
              rows, highlight={2}))
print("""
· 값은 논문 그림 1 기준 (NVIDIA A100, 양자화·추론 가속 없음).
· 초록의 "3 times faster"는 OpenVLA(3.4 → 9.0, ×2.6)를 가리키고,
  4.2절의 "7 times faster"는 ManipLLM/LLaMA-AdapterV2(1.1 → 9.0, ×8.2)를
  가리킨다. 같은 논문에서 기준이 다른 두 배수를 쓰므로, 인용할 때
  '무엇 대비 몇 배'를 반드시 붙여야 한다.
· 1.1 Hz는 한 번 움직이는 판단에 900 ms다. 사람이 컵을 옮기는 동작
  하나가 1~2초인데 그 사이 한 번밖에 못 본다는 뜻이다.
· 단, RoboMamba의 시뮬 실험은 개루프(open-loop) 단발 포즈 예측이라
  9.0 Hz가 폐루프 제어 주기로 쓰인 적은 논문에 없다. FiS-VLA의 21.9 Hz와
  직접 비교하면 안 된다(그쪽은 행동 청크·폐루프 기준).""")

# ─────────────────────────────────────────────────────────────────────────────
# 그림 — 로그-로그 스케일링
# ─────────────────────────────────────────────────────────────────────────────
X0, Y0, X1, Y1 = 80, 50, 660, 300
Ls = sorted(meas)
body = T.axes(X0, Y0, X1, Y1, "토큰 수 L (log)", "시간 (log)")
lx = [math.log2(L) for L in Ls]
lxmin, lxmax = min(lx), max(lx)
vals = [v for pair in meas.values() for v in pair]
lymin, lymax = math.log10(min(vals)), math.log10(max(vals))


def pt(L, v):
    x = X0 + (math.log2(L) - lxmin) / (lxmax - lxmin) * (X1 - X0)
    y = Y1 - (math.log10(v) - lymin) / (lymax - lymin) * (Y1 - Y0)
    return x, y


body += T.polyline([pt(L, meas[L][0]) for L in Ls], "#a8443f", 2.5)
body += T.polyline([pt(L, meas[L][1]) for L in Ls], "#2f6f4f", 2.5)
# 기준선: 완전한 L² 와 완전한 L
body += T.polyline([pt(L, meas[Ls[0]][0] * (L / Ls[0]) ** 2) for L in Ls],
                   "#c9a9a5", 1.5, dash="4 3")
body += T.polyline([pt(L, meas[Ls[0]][1] * (L / Ls[0])) for L in Ls],
                   "#a5c9b5", 1.5, dash="4 3")
for L in Ls:
    x, _ = pt(L, meas[L][0])
    body += (f'<text x="{x:.0f}" y="{Y1 + 16}" text-anchor="middle" font-size="10" '
             f'fill="#55554f">{L}</text>')
body += (f'<text x="{X1 - 150}" y="{Y0 + 14}" font-size="11" fill="#a8443f">'
         f'■ 어텐션 (점선 = 이상적 L²)</text>'
         f'<text x="{X1 - 150}" y="{Y0 + 30}" font-size="11" fill="#2f6f4f">'
         f'■ SSM (점선 = 이상적 L)</text>')
p = T.save_svg(os.path.join(OUT, "lab2_scaling.svg"), 760, 340, body,
               "어텐션 O(L²) vs 선택적 스캔 O(L) — 측정값과 이론선")
print(f"\n그림 저장: {p}")


In [ ]:
from IPython.display import SVG, display
display(SVG(filename="out/lab2_scaling.svg"))


---

## 실습 3 · '선택적' SSM이 무엇을 선택하는가

```
논문 1절: "Mamba introduces the innovative selective State Space Model (SSM),
promoting context-aware reasoning while maintaining linear complexity."

이 '선택'이 없으면 정확히 무엇을 못 하는지 보인다.
과제는 선택적 복사(selective copy): 토큰 열 중 표시(marker)가 붙은 토큰의
값 하나만 끝까지 기억해 마지막 위치에서 내놓아야 한다. 나머지는 방해물이다.

  모델 A (비선택적, S4 계열): Δ가 입력과 무관한 상수
  모델 B (선택적,  S6 = Mamba): Δ가 입력의 함수 ← 논문이 쓰는 것

두 모델에 **똑같은 크기의 상태**와 **각자에게 최적인 선형 readout**을 준다.
readout은 최소제곱으로 닫힌 해를 구하므로 "학습이 덜 됐다"는 변명이 없다.
Δ도 격자탐색으로 각 모델의 최적값을 찾아 준다. 그래도 격차가 남으면
그것은 구조의 한계다.

실행: python3 lab3_selective_gating.py    (약 12초, out/lab3_selective.svg 생성)
```

원본 스크립트: `lab3_selective_gating.py`


In [ ]:
"""실습 3 · '선택적' SSM이 무엇을 선택하는가 — 가이드 1.5 · 2.2 · 2.4

논문 1절: "Mamba introduces the innovative selective State Space Model (SSM),
promoting context-aware reasoning while maintaining linear complexity."

이 '선택'이 없으면 정확히 무엇을 못 하는지 보인다.
과제는 선택적 복사(selective copy): 토큰 열 중 표시(marker)가 붙은 토큰의
값 하나만 끝까지 기억해 마지막 위치에서 내놓아야 한다. 나머지는 방해물이다.

  모델 A (비선택적, S4 계열): Δ가 입력과 무관한 상수
  모델 B (선택적,  S6 = Mamba): Δ가 입력의 함수 ← 논문이 쓰는 것

두 모델에 **똑같은 크기의 상태**와 **각자에게 최적인 선형 readout**을 준다.
readout은 최소제곱으로 닫힌 해를 구하므로 "학습이 덜 됐다"는 변명이 없다.
Δ도 격자탐색으로 각 모델의 최적값을 찾아 준다. 그래도 격차가 남으면
그것은 구조의 한계다.

실행: python3 lab3_selective_gating.py    (약 12초, out/lab3_selective.svg 생성)
"""

import os
import sys

import numpy as np

import tinyssm as T

OUT = os.path.join(".", "out")
os.makedirs(OUT, exist_ok=True)

L, N, NTRAIN, NTEST = 32, 8, 600, 600
A = -np.arange(1, N + 1, dtype=np.float64)   # 저장소의 S4D-real 초기값 A = −(1..N)

print(T.header(3, "'선택적' SSM이 무엇을 선택하는가", "1.5 · 2.2 · 2.4"))


def make(n, seed):
    """값 열 (n,L), 표시 마스크 (n,L), 정답 (n,)"""
    g = np.random.default_rng(seed)
    vals = g.uniform(-1, 1, (n, L))
    mk = g.integers(4, L - 4, n)             # 표시 위치는 예제마다 다르다
    mask = np.zeros((n, L))
    mask[np.arange(n), mk] = 1.0
    return vals, mask, vals[np.arange(n), mk]


def final_state(vals, delta):
    """tinyssm.selective_scan과 같은 순환식(B=C=1, 채널 1개, 상태 N개)의 최종 상태.
       h_t = exp(ΔA)·h_{t−1} + Δ·x_t   ← 논문 식 (2)(4)"""
    n = vals.shape[0]
    h = np.zeros((n, N))
    for t in range(L):
        d = delta[:, t][:, None]
        h = np.exp(d * A[None]) * h + d * vals[:, t][:, None]
    return h


def best_linear_readout(Htr, ytr, Hte, yte):
    """상태에서 정답으로 가는 최적 선형 사상을 최소제곱으로 구하고 테스트 MSE."""
    Xtr = np.concatenate([Htr, np.ones((len(Htr), 1))], axis=1)
    w, *_ = np.linalg.lstsq(Xtr, ytr, rcond=None)
    Xte = np.concatenate([Hte, np.ones((len(Hte), 1))], axis=1)
    return float(np.mean((Xte @ w - yte) ** 2)), w


vtr, mtr, ytr = make(NTRAIN, 1)
vte, mte, yte = make(NTEST, 2)
var = float(np.var(yte))

print(f"""
과제  토큰 {L}개 중 '표시된' 토큰의 값 하나를 마지막 위치에서 내놓기
상태  두 모델 모두 숫자 {N}개로 고정 (A = −(1..{N}), 저장소 초기값과 같음)
평가  테스트 {NTEST}개의 MSE. 정답 분산 {var:.4f}이 '아무것도 못 배움' 기준선""")

# ── 모델 A: 비선택적. Δ 상수를 40개 후보에서 최적 선택 ──────────────────────
cands_a = np.logspace(-3, 1, 40)
curve_a = []
best_a = (float("inf"), None)
for d0 in cands_a:
    e, _ = best_linear_readout(final_state(vtr, np.full((NTRAIN, L), d0)), ytr,
                               final_state(vte, np.full((NTEST, L), d0)), yte)
    curve_a.append(e)
    if e < best_a[0]:
        best_a = (e, d0)

# ── 모델 B: 선택적. Δ_t = δ_lo + (δ_hi − δ_lo)·표시_t, 10×10 격자 ───────────
lo_grid, hi_grid = np.logspace(-3, -0.7, 10), np.logspace(-1, 1.2, 10)
best_b = (float("inf"), None, None)
for dlo in lo_grid:
    for dhi in hi_grid:
        e, _ = best_linear_readout(final_state(vtr, dlo + (dhi - dlo) * mtr), ytr,
                                   final_state(vte, dlo + (dhi - dlo) * mte), yte)
        if e < best_b[0]:
            best_b = (e, dlo, dhi)

rows = [
    ["A · 비선택적 (Δ 상수)", f"{len(cands_a)}개 후보 중 최적",
     f"Δ={best_a[1]:.4f}", f"{best_a[0]:.4f}", f"{best_a[0] / var * 100:5.1f}%"],
    ["B · 선택적 (Δ=f(표시))", f"{len(lo_grid) * len(hi_grid)}개 후보 중 최적",
     f"Δ_lo={best_b[1]:.4f} / Δ_hi={best_b[2]:.2f}",
     f"{best_b[0]:.4f}", f"{best_b[0] / var * 100:5.1f}%"],
]
print()
print(T.table(["모델", "Δ 탐색", "최적 Δ", "테스트 MSE", "정답 분산 대비"],
              rows, highlight={1}))

print(f"""
왜 비선택적 모델은 원리적으로 못 하는가
────────────────────────────────────────────────────────────────
Δ가 상수면 최종 상태는 h[j] = Σ_t Δ·exp(ΔA_j(L−1−t))·v_t 다.
즉 **위치에만 의존하는 고정 가중치로 모든 값을 더한 것** {N}개다.
선형 readout이 할 수 있는 일은 이 {N}개를 다시 섞는 것뿐이라,
결국 "v_t들의 고정 가중합"밖에 못 만든다. 그런데 정답은 예제마다
다른 위치의 값이다. 고정 가중합으로는 원리적으로 불가능하다
→ 최적값을 다 뒤져도 분산의 {best_a[0] / var * 100:.0f}%가 남는다.

선택적 모델은 표시 토큰에서만 Δ를 {best_b[2] / best_b[1]:.0f}배 키운다.
그 순간 exp(ΔA)≈0이 되어 **과거를 지우고**(경쟁자 제거) Δ·v를 크게 쓴다.
나머지 토큰에서는 Δ_lo={best_b[1]:.4f}로 작아 exp(ΔA)≈1, 즉 **그대로 보존**하며
새 입력도 거의 안 받는다. 실습 1 (3)의 '기억 반경'을 토큰마다 바꾼 것이다.
같은 상태 {N}개로 MSE {best_b[0]:.4f} — 분산의 {best_b[0] / var * 100:.1f}%.

이것이 논문이 Mamba를 고른 단 하나의 이유다. 로봇 문맥에서 그대로 읽으면
"last 20 steps: 1- open the drawer ..." 같은 긴 이력에서 지금 판단에
필요한 몇 토큰만 상태에 남기는 능력이고, 고정 크기 상태로 그걸 하니
문맥이 길어져도 비용이 늘지 않는다(실습 2).

저장소 대응 ({T.REPO})
────────────────────────────────────────────────────────────────
  Δ를 입력에서 만드는 경로 : modeling_mamba.py
      x_dbl = x_proj(conv1d_out) → dt, B, C로 split → dt_proj(dt) → softplus
  선택성이 붙는 축         : B, C가 (batch, N, L)로 토큰마다 다름
                             (S4는 (D, N) 하나로 고정 — 그것이 모델 A)
  LoRA가 건드리는 모듈     : vlm.py LinearVLM.lora()
      target_modules=["in_proj","dt_proj","x_proj","out_proj"]
      → 파인튜닝으로 '선택 규칙'을 바꾸는 지점이 정확히 dt_proj·x_proj다

주의 (이 실습의 한계)
  · 진짜 Mamba는 Δ를 표시 비트가 아니라 학습된 x_proj·dt_proj로 만든다.
    여기서는 최적 Δ 정책을 격자탐색으로 대신 찾아 '구조의 상한'을 비교했다.
  · 채널 1개·B=C=1로 줄였다. 실제는 d_inner=5120 채널이 각자 Δ를 갖는다.

해 볼 것
  · L을 64, 128로 늘리면 두 모델의 격차가 어떻게 되는가?
  · 표시를 두 개 주고 '두 번째 표시의 값'을 물으면 상태 {N}개로 되는가?
  · Δ_hi를 아주 크게(100) 주면 왜 다시 나빠지는가? (exp(ΔA)=0 → 이전 것 전멸)""")

# ─────────────────────────────────────────────────────────────────────────────
# 그림
# ─────────────────────────────────────────────────────────────────────────────
X0, Y0, X1, Y1 = 80, 54, 420, 268
body = T.axes(X0, Y0, X1, Y1, "상수 Δ (log)", "테스트 MSE")
lx = np.log10(cands_a)
ymax = var * 1.15
body += T.polyline([(X0 + (lx[i] - lx[0]) / (lx[-1] - lx[0]) * (X1 - X0),
                     Y1 - min(curve_a[i], ymax) / ymax * (Y1 - Y0))
                    for i in range(len(cands_a))], "#a8443f", 2)
yv = Y1 - var / ymax * (Y1 - Y0)
body += (f'<line x1="{X0}" y1="{yv:.0f}" x2="{X1}" y2="{yv:.0f}" stroke="#8a8a82" '
         f'stroke-dasharray="4 3"/><text x="{X0 + 6}" y="{yv - 6:.0f}" font-size="10" '
         f'fill="#55554f">정답 분산 = 아무것도 못 배움</text>')
yb = Y1 - best_b[0] / ymax * (Y1 - Y0)
body += (f'<line x1="{X0}" y1="{yb:.0f}" x2="{X1}" y2="{yb:.0f}" stroke="#2f6f4f" '
         f'stroke-width="2"/><text x="{X0 + 6}" y="{yb - 6:.0f}" font-size="10" '
         f'fill="#2f6f4f">선택적 모델 (MSE {best_b[0]:.4f})</text>')
body += (f'<text x="{X0 + 6}" y="{Y0 + 14}" font-size="11" fill="#a8443f">'
         f'■ 비선택적: Δ를 어떻게 골라도 분산 근처</text>')

# 오른쪽: 최적 선택적 Δ 프로파일
P0, P1 = 500, 840
dprof = (best_b[1] + (best_b[2] - best_b[1]) * mte)[0]
mpos = int(np.argmax(mte[0]))
body += T.axes(P0, Y0, P1, Y1, "토큰 위치", "Δ_t (선택적 모델 최적해)")
dmax = dprof.max() * 1.18
body += T.polyline([(P0 + t / (L - 1) * (P1 - P0), Y1 - dprof[t] / dmax * (Y1 - Y0))
                    for t in range(L)], "#2f6f4f", 2)
mx = P0 + mpos / (L - 1) * (P1 - P0)
body += (f'<line x1="{mx:.0f}" y1="{Y0}" x2="{mx:.0f}" y2="{Y1}" stroke="#a8443f" '
         f'stroke-dasharray="3 3"/><text x="{mx + 5:.0f}" y="{Y0 + 14}" font-size="10" '
         f'fill="#a8443f">표시 토큰 (Δ ×{best_b[2] / best_b[1]:.0f})</text>')
p = T.save_svg(os.path.join(OUT, "lab3_selective.svg"), 890, 310, body,
               "선택성이 있어야 고정 크기 상태로 '그 토큰 하나'를 기억한다")
print(f"\n그림 저장: {p}")


In [ ]:
from IPython.display import SVG, display
display(SVG(filename="out/lab3_selective.svg"))


---

## 실습 4 · 이미지를 Mamba에 넣는 방법과 '순서'의 대가

```
논문 3.2절: CLIP ViT-L로 f_v ∈ R^(B×N×1024)를 뽑고, MLP 연결기로
f_v^L ∈ R^(B×N×2560)으로 옮긴 뒤 텍스트 토큰과 이어 붙여 Mamba에 넣는다.

저장소 vlm.py LinearVLM.encode() 60행이 그 이어 붙이는 순서를 정한다.
    text_result[multi_modal] = torch.cat([vision_encoded_result, text_encoded], dim=1)
즉 **[이미지 토큰 … , 텍스트 토큰 …]** 순서다. 트랜스포머에서는 별 얘기가
아니지만 Mamba에서는 결과가 달라진다. 그 차이를 직접 확인한다.

실행: python3 lab4_vlm_token_order.py     (약 3초, out/lab4_order.svg 생성)
```

원본 스크립트: `lab4_vlm_token_order.py`


In [ ]:
"""실습 4 · 이미지를 Mamba에 넣는 방법과 '순서'의 대가 — 가이드 2.3 · 2.4

논문 3.2절: CLIP ViT-L로 f_v ∈ R^(B×N×1024)를 뽑고, MLP 연결기로
f_v^L ∈ R^(B×N×2560)으로 옮긴 뒤 텍스트 토큰과 이어 붙여 Mamba에 넣는다.

저장소 vlm.py LinearVLM.encode() 60행이 그 이어 붙이는 순서를 정한다.
    text_result[multi_modal] = torch.cat([vision_encoded_result, text_encoded], dim=1)
즉 **[이미지 토큰 … , 텍스트 토큰 …]** 순서다. 트랜스포머에서는 별 얘기가
아니지만 Mamba에서는 결과가 달라진다. 그 차이를 직접 확인한다.

실행: python3 lab4_vlm_token_order.py     (약 3초, out/lab4_order.svg 생성)
"""

import math
import os
import sys

import numpy as np

import tinyssm as T

OUT = os.path.join(".", "out")
os.makedirs(OUT, exist_ok=True)
rng = np.random.default_rng(0)

print(T.header(4, "이미지를 Mamba에 넣는 방법과 '순서'의 대가", "2.3 · 2.4"))

# ─────────────────────────────────────────────────────────────────────────────
# (1) 실제 차원으로 토큰 열 만들어 보기
# ─────────────────────────────────────────────────────────────────────────────
print("""
(1) 이미지 한 장이 토큰 몇 개가 되는가 (실제 차원)
────────────────────────────────────────────────────────────────""")
rows = []
for tag, res, patch in (("CLIP224 (논문 주 설정)", 224, 14), ("CLIP336", 336, 14),
                        ("SIGLIP256", 256, 16), ("SIGLIP384", 384, 16)):
    n_tok = (res // patch) ** 2
    rows.append([tag, f"{res}×{res}", patch, n_tok, 1024, f"{n_tok * 1024:,}"])
print(T.table(["vision.py의 encoder_type", "입력 해상도", "패치", "패치 토큰 수",
               "토큰 차원", "숫자 개수"], rows, highlight={0}))
print("""
저장소 vision.py의 vision_encoders 딕셔너리에 이 네 개가 있고,
test.sh는 --vision_encoder CLIP224를 쓴다. 논문 표 1의 'Res. 224' 행이 이것.

한 가지 더: vision.py 52행이 마지막 층이 아니라 **끝에서 두 번째 층**을 꺼낸다.
    x = self.model.get_intermediate_layers(x, n={len(self.model.blocks) - 2})[0]
마지막 층은 CLIP의 대조학습 목적(이미지 전체를 한 벡터로)에 맞춰 특화돼
국소 정보가 뭉개진다는 것이 LLaVA 계열의 경험이고, RoboMamba도 같은 선택을
했다. 논문 본문에는 없는, 코드에만 있는 사실이다(확인 2026-09-11).""")

# ─────────────────────────────────────────────────────────────────────────────
# (2) Projector — 논문의 "simple cross-modal connector"
# ─────────────────────────────────────────────────────────────────────────────
print("""
(2) 연결기(Projector)의 실제 모양과 크기
────────────────────────────────────────────────────────────────
저장소 vlm.py Projector: Linear(1024→4096) → GELU → Linear(4096→2560)
                          → GELU → Linear(2560→2560)""")
d_v, d_l = 1024, 2560
p1 = d_v * (d_v * 4) + d_v * 4
p2 = (d_v * 4) * d_l + d_l
p3 = d_l * d_l + d_l
print(T.table(["층", "모양", "파라미터"],
              [["Linear 1", f"{d_v} → {d_v * 4}", f"{p1:,}"],
               ["Linear 2", f"{d_v * 4} → {d_l}", f"{p2:,}"],
               ["Linear 3", f"{d_l} → {d_l}", f"{p3:,}"],
               ["합계", "", f"{p1 + p2 + p3:,} ({T.fmt_params(p1 + p2 + p3)})"]],
              highlight={3}))
print(f"""
논문은 이것을 그냥 "multilayer perceptron (MLP)"이라고만 쓴다. 실제로는
3층에 중간을 4배로 부풀린 {T.fmt_params(p1 + p2 + p3)} 모듈이고, Stage 1.1
정렬 사전학습에서 **이것만** 학습한다(비전 인코더·Mamba는 동결).
LLaVA의 연결기는 2층이라, 여기서 한 층 더 쓴 것은 저장소에서만 보이는 선택이다.""")

# ─────────────────────────────────────────────────────────────────────────────
# (3) 순서가 왜 문제인가 — 인과 순환의 결과
# ─────────────────────────────────────────────────────────────────────────────
print("""
(3) [이미지, 텍스트] 순서 vs [텍스트, 이미지] 순서
────────────────────────────────────────────────────────────────
Mamba는 인과(causal) 순환이다. t번째 토큰의 출력은 1..t번 토큰만 본다.
트랜스포머 디코더도 인과지만, 어텐션은 과거 토큰을 **직접 다시 읽는다**.
Mamba는 과거를 고정 크기 상태로 **압축해 들고 갈 뿐**이다. 그래서 순서가
'무엇이 무엇을 볼 수 있나'뿐 아니라 '얼마나 선명하게 보나'까지 바꾼다.""")

d_model = 48
W = T.mamba_params(d_model, seed=3)
n_img, n_txt = 16, 6
img = rng.normal(0, 1, (n_img, d_model))
txt = rng.normal(0, 1, (n_txt, d_model))

seq_it = np.concatenate([img, txt], axis=0)     # 저장소 순서
seq_ti = np.concatenate([txt, img], axis=0)     # 뒤집은 순서
y_it, _ = T.mamba_block(seq_it, W)
y_ti, _ = T.mamba_block(seq_ti, W)

# 마지막 토큰의 출력이 각 입력 토큰에 얼마나 민감한가 (수치 미분)
def sensitivity(seq, eps=1e-4):
    base, _ = T.mamba_block(seq, W)
    last = base[-1].copy()
    out = np.zeros(len(seq))
    for i in range(len(seq)):
        s = seq.copy()
        s[i] += eps
        y, _ = T.mamba_block(s, W)
        out[i] = np.abs(y[-1] - last).sum() / eps
    return out

s_it = sensitivity(seq_it)
s_ti = sensitivity(seq_ti)
img_it, txt_it = s_it[:n_img].mean(), s_it[n_img:].mean()
img_ti, txt_ti = s_ti[n_txt:].mean(), s_ti[:n_txt].mean()

print(T.table(["토큰 순서", "마지막 토큰", "이미지 토큰 민감도(평균)",
               "텍스트 토큰 민감도(평균)", "이미지/텍스트"],
              [[f"[이미지{n_img}, 텍스트{n_txt}] ← 저장소", "텍스트 끝",
                f"{img_it:.4f}", f"{txt_it:.4f}", f"{img_it / txt_it:.3f}"],
               [f"[텍스트{n_txt}, 이미지{n_img}]", "이미지 끝",
                f"{img_ti:.4f}", f"{txt_ti:.4f}", f"{img_ti / txt_ti:.3f}"]],
              highlight={0}))
print(f"""
읽는 법
  · 어느 순서든 **뒤에 온 토큰이 훨씬 큰 영향**을 준다. 앞쪽 토큰의 정보는
    exp(ΔA)를 여러 번 곱하며 지수적으로 옅어진다(실습 1의 기억 반경).
  · 저장소 순서([이미지, 텍스트])에서는 지시문이 상태에 가장 가깝게 남고,
    이미지 패치 {n_img}개는 그보다 멀어진다. 답을 만드는 마지막 위치에서 보면
    '지시는 선명하고, 이미지는 요약되어' 들어 있다.
  · 뒤집으면 반대가 된다. 이미지가 선명하고 지시가 옅어진다.
  · 트랜스포머라면 마지막 토큰이 어텐션으로 아무 패치나 다시 정확히 읽을 수
    있으므로 이 비대칭이 훨씬 약하다. **이것이 Mamba를 쓴 대가다.**

이 실습은 무작위 가중치의 축소 모형이라 절대 수치를 일반화하면 안 된다.
읽어야 할 것은 '순서에 따라 비대칭이 생긴다'는 방향성 하나다.

스터디에서 따져 볼 것
  · 논문 표 1에서 RoboMamba가 POPE(환각) 86.3·GQA 64.2로 좋은 반면
    MME(1297.2)·MMBench(60.9)는 LLaVA1.5(1510.7·64.3)보다 낮다.
    '이미지를 앞에 두고 압축해 통과시키는' 구조와 관계가 있을까?
  · 논문에 이미지/텍스트 순서를 바꾼 소거 실험은 **없다**(부록 C 확인).
    RoboMamba 후속인 FiS-VLA가 트랜스포머(LLaMA2)로 돌아간 이유 중 하나로
    이 비대칭을 꼽을 수 있는가? — 부스 질문 후보.""")

# ─────────────────────────────────────────────────────────────────────────────
# (4) 전체 입력 열 조립
# ─────────────────────────────────────────────────────────────────────────────
print(f"""
(4) 논문 주 설정의 실제 입력 열 (CLIP224 + mamba-2.8b)
────────────────────────────────────────────────────────────────
  이미지 224×224 → ViT-L/14 패치 16×16 = 256 토큰 × 1024차원
      ↓ Projector (1024→4096→2560→2560), {T.fmt_params(p1 + p2 + p3)}
  이미지 토큰 256 × 2560
  지시문 "Predict the contact point and orientation for pulling the {{object}}"
      ↓ Mamba 토크나이저 (GPT-NeoX 계열, vocab 50280)
  텍스트 토큰 약 15 × 2560
      ↓ torch.cat([vision, text], dim=1)      ← vlm.py 60행
  입력 열 약 271 × 2560  →  Mamba 블록 64개  →  출력 열 271 × 2560
      ↓ (추론) lm_head → 다음 토큰       ... 언어 답변 (reasoning)
      ↓ (조작)  정책 헤드                ... 실습 5로""")

# ─────────────────────────────────────────────────────────────────────────────
# 그림
# ─────────────────────────────────────────────────────────────────────────────
X0, Y0, X1 = 70, 60, 820
body = ""
for k, (lab, s, split, order) in enumerate((
        (f"[이미지 {n_img} → 텍스트 {n_txt}]  ← 저장소 순서", s_it, n_img, "it"),
        (f"[텍스트 {n_txt} → 이미지 {n_img}]", s_ti, n_txt, "ti"))):
    base = Y0 + k * 128
    w = (X1 - X0) / len(s)
    smax = max(s_it.max(), s_ti.max())
    body += (f'<text x="{X0}" y="{base - 10}" font-size="12" font-weight="600" '
             f'fill="#1a1a18">{lab}</text>')
    for i, v in enumerate(s):
        is_img = (i >= split) if order == "ti" else (i < split)
        col = "#3f6fa8" if is_img else "#b8863f"
        h = max(2.0, v / smax * 80)
        body += (f'<rect x="{X0 + i * w:.1f}" y="{base + 84 - h:.1f}" '
                 f'width="{w - 1.5:.1f}" height="{h:.1f}" fill="{col}" opacity="0.85"/>')
    body += (f'<line x1="{X0}" y1="{base + 84}" x2="{X1}" y2="{base + 84}" '
             f'stroke="#8a8a82"/>')
body += (f'<text x="{X0}" y="{Y0 + 262}" font-size="11" fill="#3f6fa8">'
         f'■ 이미지 토큰</text>'
         f'<text x="{X0 + 110}" y="{Y0 + 262}" font-size="11" fill="#b8863f">'
         f'■ 텍스트 토큰</text>'
         f'<text x="{X0 + 240}" y="{Y0 + 262}" font-size="11" fill="#55554f">'
         f'막대 높이 = 마지막 토큰 출력이 그 입력 토큰에 얼마나 민감한가</text>')
p = T.save_svg(os.path.join(OUT, "lab4_order.svg"), 880, 300, body,
               "Mamba는 '뒤에 온 토큰'을 선명하게 본다 — 순서가 설계 결정이 된다")
print(f"\n그림 저장: {p}")


In [ ]:
from IPython.display import SVG, display
display(SVG(filename="out/lab4_order.svg"))


---

## 실습 5 · 정책 헤드 하나로 조작을 배우기

```
논문 3.4절의 주장을 그대로 재현한다.
  "once RoboMamba possesses sufficient reasoning capability, it can acquire
   pose prediction skills with minimal fine-tuning parameters and time."

저장소 manip.py의 구조를 그대로 쓴다.
  · SpecialMLP: Linear(h, h/2) → ReLU → Linear(h/2, h/4) → ReLU → Linear(h/4, out, bias=False)
  · head1 → 2개 (접촉점 x, y)          ... 논문 식 (5) L1 손실
  · head2 → 6개 (6D 회전 표현)          ... 논문 식 (6) 측지 손실
  · 백본은 전부 동결. 헤드만 학습.

세 가지를 확인한다.
  (1) 동결 백본 + 작은 헤드가 실제로 빠르게 학습되는가
  (2) 논문 그림 3 b)의 주장 — 백본의 '이해 수준'이 조작 정확도를 좌우하는가
  (3) 왜 회전을 9개 숫자가 아니라 6개로 내놓는가

실행: python3 lab5_policy_head_6d.py      (약 25초, out/lab5_head.svg 생성)
```

원본 스크립트: `lab5_policy_head_6d.py`


In [ ]:
"""실습 5 · 정책 헤드 하나로 조작을 배우기 — 가이드 2.5 · 2.7

논문 3.4절의 주장을 그대로 재현한다.
  "once RoboMamba possesses sufficient reasoning capability, it can acquire
   pose prediction skills with minimal fine-tuning parameters and time."

저장소 manip.py의 구조를 그대로 쓴다.
  · SpecialMLP: Linear(h, h/2) → ReLU → Linear(h/2, h/4) → ReLU → Linear(h/4, out, bias=False)
  · head1 → 2개 (접촉점 x, y)          ... 논문 식 (5) L1 손실
  · head2 → 6개 (6D 회전 표현)          ... 논문 식 (6) 측지 손실
  · 백본은 전부 동결. 헤드만 학습.

세 가지를 확인한다.
  (1) 동결 백본 + 작은 헤드가 실제로 빠르게 학습되는가
  (2) 논문 그림 3 b)의 주장 — 백본의 '이해 수준'이 조작 정확도를 좌우하는가
  (3) 왜 회전을 9개 숫자가 아니라 6개로 내놓는가

실행: python3 lab5_policy_head_6d.py      (약 25초, out/lab5_head.svg 생성)
"""

import os
import sys

import numpy as np

import tinyssm as T

OUT = os.path.join(".", "out")
os.makedirs(OUT, exist_ok=True)

HID = 256          # 축소판 (실제 llm.hidden_size = 2560)
NTRAIN, NTEST = 1500, 600
STEPS, BATCH = 700, 64

print(T.header(5, "정책 헤드 하나로 조작을 배우기", "2.5 · 2.7"))

# ─────────────────────────────────────────────────────────────────────────────
# 장난감 '관절 물체 당기기' 데이터
# ─────────────────────────────────────────────────────────────────────────────


def make_scene(n, seed, y_rule="canonical"):
    """잠재 상태 z = [접촉점 x, y, 표면 법선 3개]와 정답 포즈.

    논문 4.1절의 데이터 수집 규칙:
      "randomly select a contact point p on the movable part and orient the
       end-effector's z-axis opposite to its normal vector, with a random
       y-axis direction"

    z축은 법선이 정해 주지만 **y축은 임의**라고 적혀 있다. 이 한 줄이
    회전 학습에 무슨 뜻인지가 이 실습 (2)의 주제다. 두 규칙을 다 만든다.
      y_rule="canonical" : y축을 월드 업벡터에서 결정 (장면의 함수)
      y_rule="random"    : 논문 문장 그대로 임의 (장면의 함수가 아님)
    """
    g = np.random.default_rng(seed)
    pos = g.uniform(0.15, 0.85, (n, 2))                  # 접촉점 (이미지 좌표)
    nrm = g.normal(0, 1, (n, 3))
    nrm /= np.linalg.norm(nrm, axis=1, keepdims=True)    # 표면 법선
    z_axis = -nrm                                        # 그리퍼 z축 = 법선 반대
    if y_rule == "canonical":
        tmp = np.tile(np.array([0.0, 0.0, 1.0]), (n, 1))  # 월드 업
        deg = np.abs((tmp * z_axis).sum(1)) > 0.98        # z와 거의 평행하면
        tmp[deg] = np.array([0.0, 1.0, 0.0])              # 대체 기준축
        tmp = tmp + g.normal(0, 0.02, (n, 3))             # 수집 잡음
    else:
        tmp = g.normal(0, 1, (n, 3))
    y_axis = tmp - (tmp * z_axis).sum(1, keepdims=True) * z_axis
    y_axis /= np.linalg.norm(y_axis, axis=1, keepdims=True)
    x_axis = np.cross(y_axis, z_axis)
    R = np.stack([x_axis, y_axis, z_axis], axis=-1)      # (n,3,3) 열이 축
    return np.concatenate([pos, nrm], axis=1), pos, R


def backbone_features(z, quality, seed):
    """동결 백본이 내놓는 '전역 토큰'을 흉내 낸다.

    quality = 장면을 얼마나 이해했는가(= 논문이 말하는 reasoning ability).
      1.0 → 잠재 상태 5개가 모두 특징 안에 선형으로 들어 있다
      0.5 → 절반만 들어 있고 나머지 자리는 잡음
      0.0 → 전부 잡음 (장면을 전혀 못 읽음)
    """
    g = np.random.default_rng(seed)
    n, dz = z.shape
    keep = int(round(dz * quality))
    latent = np.concatenate([z[:, :keep], g.normal(0, 1, (n, dz - keep))], axis=1)
    M = np.random.default_rng(1234).normal(0, 1, (dz, HID)) / np.sqrt(dz)
    return latent @ M + g.normal(0, 0.05, (n, HID))


# ─────────────────────────────────────────────────────────────────────────────
# 헤드 (저장소 manip.py two_mlp와 같은 구조)
# ─────────────────────────────────────────────────────────────────────────────


def z_axis_error_deg(R_pred, R_gt):
    """그리퍼 z축(접근 방향)만의 각오차. 흡착 그리퍼는 z축 주위 회전이
    결과를 바꾸지 않으므로, 논문의 성공률에 실제로 걸리는 양은 이것이다."""
    zp, zg = R_pred[:, :, 2], R_gt[:, :, 2]
    return np.degrees(np.arccos(np.clip((zp * zg).sum(1), -1, 1)))


def geo_grad(d6, R_gt, eps=1e-5):
    """논문 식 (6)의 기울기. 6개 숫자에 대해서만 수치미분하고 나머지는 해석적.

    저장소 loss_6d_rot()를 그대로 손실로 쓰면서도 역전파가 가능한 방법이다.
    (6D → 3×3 Gram-Schmidt → arccos 경로를 손으로 미분하지 않아도 된다)
    """
    n = len(d6)
    base = T.geodesic_loss(T.gram_schmidt_6d(d6), R_gt)
    g = np.zeros_like(d6)
    for k in range(6):
        d = d6.copy()
        d[:, k] += eps
        g[:, k] = (T.geodesic_loss(T.gram_schmidt_6d(d), R_gt) - base) / eps
    return g / n, float(base.mean())


def train_heads(quality, seed=0, log=None, y_rule="canonical"):
    rng = np.random.default_rng(seed)
    ztr, ptr, Rtr = make_scene(NTRAIN, 10 + seed, y_rule)
    zte, pte, Rte = make_scene(NTEST, 500 + seed, y_rule)
    Ftr = backbone_features(ztr, quality, 20 + seed)
    Fte = backbone_features(zte, quality, 20 + seed)

    head1 = T.SpecialMLP(HID, 2, rng)      # 접촉점
    head2 = T.SpecialMLP(HID, 6, rng)      # 6D 회전
    opt = T.Adam(head1.params() + head2.params(), lr=3e-4)

    hist = []
    for s in range(STEPS):
        idx = rng.integers(0, NTRAIN, BATCH)
        f, pg, Rg = Ftr[idx], ptr[idx], Rtr[idx]

        # 식 (5) 위치: L1
        pred_p = head1(f)
        diff = pred_p - pg
        l_pos = float(np.abs(diff).mean())
        g_pos = np.sign(diff) / diff.size

        # 식 (6) 방향: 측지 거리
        pred_d = head2(f)
        g_dir, l_dir = geo_grad(pred_d, Rg)

        opt.zero_grad()
        head1.backward(g_pos)
        head2.backward(g_dir)
        opt.step()
        hist.append((l_pos, l_dir))
        if log and s % 175 == 0:
            print(f"    quality={quality:.1f}  step {s:4d}  "
                  f"L_pos {l_pos:.4f}  L_dir {np.degrees(l_dir):5.1f}°")

    # 테스트
    pp = head1(Fte)
    Rp = T.gram_schmidt_6d(head2(Fte))
    ang = np.degrees(T.geodesic_loss(Rp, Rte))
    zerr = z_axis_error_deg(Rp, Rte)
    perr = np.abs(pp - pte).mean(axis=1)
    succ = float(np.mean((perr < 0.1) & (zerr < 30.0)))
    return dict(quality=quality, hist=hist, pos_err=float(perr.mean()),
                ang=float(ang.mean()), zerr=float(zerr.mean()), succ=succ,
                y_rule=y_rule, n_params=head1.n_params() + head2.n_params())


# ─────────────────────────────────────────────────────────────────────────────
# (1) 동결 백본 + 작은 헤드
# ─────────────────────────────────────────────────────────────────────────────
print(f"""
(1) 백본 동결, 헤드만 학습 — 얼마나 빨리 배우는가
────────────────────────────────────────────────────────────────
헤드 구조는 저장소 manip.py와 동일(SpecialMLP ×2). 여기서는 hidden={HID}
축소판이고 실제는 2560이다. 학습 {STEPS}스텝 · 배치 {BATCH}.
손실은 논문 식 (5) L1(위치) + 식 (6) 측지거리(회전), 저장소와 같다.
""")
full = train_heads(1.0, log=True)
print(f"""
  최종  접촉점 평균오차   {full['pos_err']:.4f}   (이미지 좌표 0~1 기준)
        회전 전체 오차    {full['ang']:.2f}°   (식 6, 3×3 행렬 전체)
        그리퍼 z축 오차   {full['zerr']:.2f}°   (접근 방향만)
        성공률            {full['succ'] * 100:.1f}%   (위치<0.1 & z축<30°)
        학습 파라미터     {T.fmt_params(full['n_params'])}

논문 3.4절의 "a few dozen minutes on a single A100"이 이 구조 때문이다.
백본이 동결이면 역전파가 헤드 안에서만 돌고 옵티마이저 상태도 헤드 것만
들고 있으면 된다. 여기서는 노트북 CPU로 10초.""")

# ─────────────────────────────────────────────────────────────────────────────
# (2) 논문 4.1절의 'random y-axis' 한 줄이 뜻하는 것
# ─────────────────────────────────────────────────────────────────────────────
print("""
(2) 정답 회전이 애초에 예측 가능한가 — 논문 4.1절 데이터 수집 규칙 읽기
────────────────────────────────────────────────────────────────
논문 4.1절:
  "randomly select a contact point p on the movable part and orient the
   end-effector's z-axis opposite to its normal vector, **with a random
   y-axis direction** to interact with the object"

z축은 표면 법선이 정해 준다. 그런데 y축은 '임의'다. 그러면 정답 3×3 행렬은
장면의 함수가 아니고, z축 주위 회전각만큼 예측 불가능한 성분을 담는다.
식 (6)의 측지 손실은 그 성분까지 맞히라고 요구한다. 무슨 일이 생기는가.""")
rand_y = train_heads(1.0, seed=0, y_rule="random")
rows = [
    ["y축을 장면이 결정 (canonical)", f"{full['pos_err']:.4f}",
     f"{full['ang']:6.2f}°", f"{full['zerr']:6.2f}°", f"{full['succ'] * 100:5.1f}%"],
    ["y축을 임의로 (논문 문장 그대로)", f"{rand_y['pos_err']:.4f}",
     f"{rand_y['ang']:6.2f}°", f"{rand_y['zerr']:6.2f}°",
     f"{rand_y['succ'] * 100:5.1f}%"],
]
print(T.table(["데이터 수집 규칙", "접촉점 오차", "회전 전체 오차(식 6)",
               "z축 오차", "성공률"], rows, highlight={1}))
g0 = np.random.default_rng(3)
_, _, Ra = make_scene(600, 77, "random")
_, _, Rb = make_scene(600, 78, "random")
chance = float(np.degrees(T.geodesic_loss(Ra, Rb)).mean())
print(f"""
읽는 법
  · 임의 y축에서는 회전 전체 오차가 {rand_y['ang']:.1f}°에 멈춘다. 무작위 회전
    두 개 사이의 평균 거리가 {chance:.1f}°이니 **거의 학습이 안 된 수준**이다.
    당연하다. 정답의 그 성분은 장면에 정보가 없다. 학습은 잡음과 싸운다.
  · 그런데 **z축 오차는 {rand_y['zerr']:.1f}°까지 내려간다.** 예측 가능한 부분은
    제대로 배운 것이다. 성공률도 {rand_y['succ'] * 100:.0f}%로 거의 안 떨어진다.
  · 왜 성공률이 안 떨어지는가: 논문의 액추에이터는 **흡착 그리퍼**(4.1절,
    부록 D는 실기에서 양면테이프를 붙여 흡착으로 바꿨다고 적는다)다.
    빨판은 z축 주위로 돌려도 결과가 같다. 그리고 논문의 성공 판정은
    "물체 관절 상태 변화가 0.1 m 초과"(4.1절)이므로 z축만 맞으면 된다.

여기서 배울 것 (스터디용)
  · 논문이 회전 오차(도)를 표로 내지 않고 성공률만 보고하는 데는 이유가 있다.
    이 설정에서 식 (6)을 지표로 쓰면 상한이 막혀 있어 모델 비교가 안 된다.
  · 반대로, 정밀한 자세가 필요한 과제(2지 그리퍼로 손잡이를 잡는 등)로
    옮기면 이 데이터 수집 규칙 자체를 바꿔야 한다.
  · 논문 5절 한계에 이 얘기는 없다. '밝히지 않은 한계' 후보다.""")

# ─────────────────────────────────────────────────────────────────────────────
# (3) 논문 그림 3 b) 재현 — 백본의 이해 수준이 조작 정확도를 좌우한다
# ─────────────────────────────────────────────────────────────────────────────
print("""
(3) 백본이 장면을 얼마나 읽었나에 따라 (논문 그림 3 b) 구조 재현)
────────────────────────────────────────────────────────────────""")
runs = [full] + [train_heads(q, seed=i + 1) for i, q in enumerate((0.6, 0.4, 0.0))]
runs.sort(key=lambda r: -r["quality"])
rows = [[f"{r['quality']:.1f}", f"{r['pos_err']:.4f}", f"{r['zerr']:6.2f}°",
         f"{r['succ'] * 100:5.1f}%", T.fmt_params(r["n_params"])] for r in runs]
print(T.table(["백본 이해 수준", "접촉점 오차", "z축 오차", "성공률",
               "학습 파라미터"], rows, highlight={0}))
print("""
헤드 크기는 네 줄 모두 같다. 달라진 것은 **백본이 장면을 얼마나 담고
있는가**뿐인데 성공률이 그에 따라 움직인다. 논문 그림 3 b)와 같은 모양이다.
  OpenFlamingo 0.26 / LLaMA-AdapterV2 0.46 / Ours-1.4B 0.39
  / Ours-2.7B(co-training 없음) 0.61 / Ours-2.7B 0.63     (seen 성공률)
논문의 결론 문장 그대로: "fine-tuning an MLLM to learn robot skills does not
require extensive resources; it only requires that the MLLM possesses strong
robotic-related reasoning abilities."

주의: 논문 그림 3 b)는 서로 다른 실제 MLLM 네 개를 비교한 것이고 여기서는
같은 백본의 '정보량'만 인공적으로 줄였다. 인과 방향(이해 → 조작)을 보여 주는
장난감이지 논문 수치를 재현한 것이 아니다.""")

# ─────────────────────────────────────────────────────────────────────────────
# (4) 왜 6개 숫자인가
# ─────────────────────────────────────────────────────────────────────────────
print("""
(4) 회전을 9개가 아니라 6개로 내놓는 이유
────────────────────────────────────────────────────────────────
논문 식 (6)은 a_dir ∈ R^(3×3) 회전행렬을 쓴다고 적었지만, 저장소
manip.py의 head2는 **6개**를 내놓고 bgs()(Gram-Schmidt)로 3×3을 만든다.
신경망이 9개 숫자를 그냥 뱉으면 그것이 회전행렬일 보장이 없다.""")
g = np.random.default_rng(0)
raw9 = g.normal(0, 1, (2000, 9)).reshape(-1, 3, 3)
err9 = np.abs(np.einsum("mij,mik->mjk", raw9, raw9)
              - np.eye(3)[None]).max(axis=(1, 2))
det9 = np.linalg.det(raw9)
raw6 = g.normal(0, 1, (2000, 6))
R6 = T.gram_schmidt_6d(raw6)
err6 = np.abs(np.einsum("mij,mik->mjk", R6, R6) - np.eye(3)[None]).max(axis=(1, 2))
det6 = np.linalg.det(R6)
print(T.table(["표현", "직교성 오차 max|RᵀR−I| 평균", "det(R) 평균",
               "유효한 회전 비율"],
              [["9개 숫자 그대로", f"{err9.mean():.4f}", f"{det9.mean():+.4f}",
                f"{np.mean(err9 < 1e-6) * 100:.1f}%"],
               ["6개 + Gram-Schmidt", f"{err6.mean():.2e}", f"{det6.mean():+.4f}",
                f"{np.mean(err6 < 1e-6) * 100:.1f}%"]], highlight={1}))
print(f"""
무작위 6개 숫자를 Gram-Schmidt에 넣으면 **항상** det=+1인 정규직교행렬이
나온다. 9개를 그냥 쓰면 회전이 아닌 행렬이 나와 식 (6)의 arccos((tr−1)/2)가
정의역을 벗어난다. 저장소가 clamp(−1+1e−6, 1−1e−6)를 두는 이유이기도 하다.
부작용: 완벽히 맞혀도 손실이 0이 아니라 {np.degrees(np.arccos(1 - 1e-6)):.3f}°에서 바닥을 친다.

저장소 대응 ({T.REPO})
  SpecialMLP            : manip.py 18~30행  Linear(h,h/2)→ReLU→Linear(h/2,h/4)→ReLU→Linear(h/4,out,bias=False)
  head1/head2 (two_mlp) : manip.py 42~43행, xavier_uniform_ 초기화 44~53행
  전역 토큰 만드는 곳   : manip.py 82행
      res = (res[:, vision_encoded.shape[1]] + res[:, -1]) / 2
      → 논문 그림 2 설명은 "global token ... generated through a pooling
        operation from the language output tokens"라 쓰지만, 실제 two_mlp
        경로는 **이미지 토큰 직후 토큰과 마지막 토큰 딱 두 개의 평균**이다.
        AdaptiveAvgPool1d(1)는 ssm+mlp 경로에서만 쓰인다. → 논문·코드 상충
  lm_head 우회          : manip.py 64~66행  self.llm.mamba.lm_head = nn.Identity()
      → .logits이 어휘 점수(50280)가 아니라 2560차원 은닉상태가 되게 하는 기법
  6D → 회전행렬         : manip.py 116~122행 bgs()
  측지 손실 식 (6)      : manip.py 125~130행 bgdR()
  위치 출력이 2개인 근거: manip.py 42행 SpecialMLP(hidden, 2)
      → 논문 3.4절 "RoboMamba only predicts the 2D position (x, y) of the
        contact pixel, which is then translated into 3D space using depth"

해 볼 것
  · head1/head2를 하나로 합치면(저장소 head_type='mlp', 출력 8개) 어떻게 되는가?
    논문 표 6은 63.7% → 62.1%로 거의 같다고 한다.
  · 위치 손실을 L1(식 5)에서 MSE로 바꾸면 접촉점 오차가 어떻게 변하는가?
  · y_rule="random"에서 손실을 z축 각오차만으로 바꾸면 수렴이 빨라지는가?""")

# ─────────────────────────────────────────────────────────────────────────────
# 그림
# ─────────────────────────────────────────────────────────────────────────────
X0, Y0, X1, Y1 = 80, 56, 400, 270
body = T.axes(X0, Y0, X1, Y1, "학습 스텝", "회전 전체 오차 (도, 식 6)")
amax = 180.0
for r, col, lab in ((full, "#2f6f4f", "y축 = 장면이 결정"),
                    (rand_y, "#a8443f", "y축 = 임의 (논문 규칙)")):
    pts = [(X0 + i / STEPS * (X1 - X0),
            Y1 - min(np.degrees(v[1]), amax) / amax * (Y1 - Y0))
           for i, v in enumerate(r["hist"])]
    body += T.polyline(pts, col, 1.8)
yc = Y1 - chance / amax * (Y1 - Y0)
body += (f'<line x1="{X0}" y1="{yc:.0f}" x2="{X1}" y2="{yc:.0f}" stroke="#8a8a82" '
         f'stroke-dasharray="4 3"/><text x="{X0 + 6}" y="{yc - 6:.0f}" font-size="10" '
         f'fill="#55554f">무작위 회전 사이 평균 거리 {chance:.0f}°</text>')
body += (f'<text x="{X0 + 8}" y="{Y0 + 14}" font-size="11" fill="#2f6f4f">'
         f'■ y축 = 장면이 결정 → {full["ang"]:.1f}°</text>'
         f'<text x="{X0 + 8}" y="{Y0 + 29}" font-size="11" fill="#a8443f">'
         f'■ y축 = 임의 → {rand_y["ang"]:.1f}° (바닥)</text>')

P0, P1 = 490, 840
body += T.axes(P0, Y0, P1, Y1, "백본 이해 수준", "성공률")
w = (P1 - P0 - 40) / len(runs) * 0.55
for i, r in enumerate(sorted(runs, key=lambda x: x["quality"])):
    h = r["succ"] * (Y1 - Y0)
    x = P0 + 24 + i * (P1 - P0 - 44) / len(runs)
    body += (f'<rect x="{x:.0f}" y="{Y1 - h:.0f}" width="{w:.0f}" height="{h:.0f}" '
             f'fill="#2f6f4f" opacity="{0.35 + 0.16 * i:.2f}"/>'
             f'<text x="{x + w / 2:.0f}" y="{Y1 - h - 6:.0f}" text-anchor="middle" '
             f'font-size="10" fill="#1a1a18">{r["succ"] * 100:.0f}%</text>'
             f'<text x="{x + w / 2:.0f}" y="{Y1 + 16:.0f}" text-anchor="middle" '
             f'font-size="10" fill="#55554f">{r["quality"]:.1f}</text>')
p = T.save_svg(os.path.join(OUT, "lab5_head.svg"), 890, 315, body,
               "왼쪽: 정답에 예측 불가 성분이 있으면 손실은 바닥에 멈춘다 · 오른쪽: 논문 그림 3 b) 구조")
print(f"\n그림 저장: {p}")


In [ ]:
from IPython.display import SVG, display
display(SVG(filename="out/lab5_head.svg"))


---

## 실습 6 · 논문의 '0.1%'를 직접 세어 보기

```
논문의 핵심 효율 주장은 숫자 하나다.
  "The fine-tuned policy head constitutes only 0.1% of the model parameters,
   which is 10 times smaller than existing robotic VLA approaches." (1절)
  부록 표 6: MLP×2 = 3.7M (0.11%) / MLP×1 = 1.8M (0.05%)
            / (SSM block+MLP)×2 = 45.2M (1.3%)

저장소 코드의 층 정의에서 파라미터를 직접 세어 이 숫자를 검증한다.
곱셈과 덧셈만 쓰므로 PyTorch도 체크포인트도 필요 없다.

실행: python3 lab6_param_audit.py     (1초 미만)
```

원본 스크립트: `lab6_param_audit.py`


In [ ]:
"""실습 6 · 논문의 '0.1%'를 직접 세어 보기 — 가이드 2.5 · 2.8

논문의 핵심 효율 주장은 숫자 하나다.
  "The fine-tuned policy head constitutes only 0.1% of the model parameters,
   which is 10 times smaller than existing robotic VLA approaches." (1절)
  부록 표 6: MLP×2 = 3.7M (0.11%) / MLP×1 = 1.8M (0.05%)
            / (SSM block+MLP)×2 = 45.2M (1.3%)

저장소 코드의 층 정의에서 파라미터를 직접 세어 이 숫자를 검증한다.
곱셈과 덧셈만 쓰므로 PyTorch도 체크포인트도 필요 없다.

실행: python3 lab6_param_audit.py     (1초 미만)
"""

import os
import sys

import tinyssm as T

print(T.header(6, "논문의 '0.1%'를 직접 세어 보기", "2.5 · 2.8"))

# ─────────────────────────────────────────────────────────────────────────────
# mamba-2.8b 하이퍼파라미터 (state-spaces/mamba-2.8b-hf config.json, 확인 2026-09-11)
# ─────────────────────────────────────────────────────────────────────────────
D_MODEL, N_LAYER, D_STATE, D_CONV, EXPAND = 2560, 64, 16, 4, 2
DT_RANK, VOCAB = 160, 50280
D_INNER = EXPAND * D_MODEL          # 5120

print(f"""
(1) Mamba 블록 하나의 파라미터 — MyMamba.__init__()의 층을 그대로 센다
────────────────────────────────────────────────────────────────
d_model={D_MODEL}  d_inner={D_INNER}  d_state(N)={D_STATE}
d_conv={D_CONV}  dt_rank={DT_RANK}  n_layer={N_LAYER}  vocab={VOCAB}""")

blk = [
    ("in_proj", f"Linear({D_MODEL} → {2 * D_INNER}), bias=False",
     D_MODEL * 2 * D_INNER),
    ("conv1d", f"Conv1d({D_INNER}, groups={D_INNER}, k={D_CONV}) + bias",
     D_INNER * D_CONV + D_INNER),
    ("x_proj", f"Linear({D_INNER} → {DT_RANK + 2 * D_STATE}), bias=False",
     D_INNER * (DT_RANK + 2 * D_STATE)),
    ("dt_proj", f"Linear({DT_RANK} → {D_INNER}) + bias",
     DT_RANK * D_INNER + D_INNER),
    ("A_log", f"({D_INNER}, {D_STATE})", D_INNER * D_STATE),
    ("D", f"({D_INNER},)", D_INNER),
    ("out_proj", f"Linear({D_INNER} → {D_MODEL}), bias=False", D_INNER * D_MODEL),
    ("norm (RMSNorm)", f"({D_MODEL},)", D_MODEL),
]
blk_total = sum(n for _, _, n in blk)
rows = [[k, d, f"{n:,}", f"{n / blk_total * 100:5.1f}%"] for k, d, n in blk]
rows.append(["블록 합계", "", f"{blk_total:,}", "100.0%"])
print(T.table(["층", "모양", "파라미터", "블록 내 비중"], rows,
              highlight={len(rows) - 1}))
print(f"""
블록 하나 {T.fmt_params(blk_total)}. in_proj 하나가 블록의 {26214400 / blk_total * 100:.0f}%를 먹는다
(d_model → 2×d_inner = 4배 확장). 어텐션이 없으니 QKV도 없다.
실제 값 대조: transformers MambaForCausalLM(config)로 세면 블록당
41,244,160개 — 위 합계와 정확히 같다 (확인 2026-09-11).""")

# ─────────────────────────────────────────────────────────────────────────────
# (2) 전체 모델
# ─────────────────────────────────────────────────────────────────────────────
llm_total = blk_total * N_LAYER + VOCAB * D_MODEL + D_MODEL   # lm_head는 임베딩과 공유
clip = 303_179_776            # timm vit_large_patch14_clip_224.openai, num_classes=0
proj = (1024 * 4096 + 4096) + (4096 * 2560 + 2560) + (2560 * 2560 + 2560)
model_total = llm_total + clip + proj

print("""
(2) RoboMamba 전체 (CLIP224 + mamba-2.8b)
────────────────────────────────────────────────────────────────""")
print(T.table(["부품", "내용", "파라미터", "비중"],
              [["Mamba LLM", f"블록 {N_LAYER}개 + 임베딩 {VOCAB}×{D_MODEL} (lm_head 공유)",
                f"{llm_total:,}", f"{llm_total / model_total * 100:5.1f}%"],
               ["비전 인코더", "CLIP ViT-L/14@224 (동결, 학습 안 함)",
                f"{clip:,}", f"{clip / model_total * 100:5.1f}%"],
               ["Projector", "Linear(1024→4096→2560→2560)",
                f"{proj:,}", f"{proj / model_total * 100:5.1f}%"],
               ["합계", "", f"{model_total:,} ({T.fmt_params(model_total)})", "100.0%"]],
              highlight={3}))
print(f"""
논문 4절은 "RoboMamba, with only 3.2B parameters"라 쓴다. 위 계산은
{T.fmt_params(model_total)}. 표 1·2의 'LLM 2.7B'는 Mamba만, 그림 1의 '2.8B'는
체크포인트 이름(mamba-2.8b)이다. **같은 모델을 세는 세 가지 방식**이므로
인용할 때 무엇을 센 숫자인지 밝혀야 한다.""")

# ─────────────────────────────────────────────────────────────────────────────
# (3) 정책 헤드 — 논문 표 6 대조
# ─────────────────────────────────────────────────────────────────────────────
print("""
(3) 정책 헤드: 저장소 코드로 센 값 vs 논문 표 6
────────────────────────────────────────────────────────────────
저장소 manip.py SpecialMLP(inp, oup):
    fc1 Linear(inp,   inp//2)             + bias
    fc2 Linear(inp//2, inp//4)            + bias
    fc3 Linear(inp//4, oup, bias=False)""")


def special_mlp(inp, oup, div1=2, div2=4):
    h1, h2 = inp // div1, inp // div2
    return (inp * h1 + h1) + (h1 * h2 + h2) + (h2 * oup)


code_2 = special_mlp(D_MODEL, 2)
code_6 = special_mlp(D_MODEL, 6)
code_two = code_2 + code_6
code_one = special_mlp(D_MODEL, 8)
# 논문 수치와 맞는 대안 축소비: inp//4, inp//8
alt_2 = special_mlp(D_MODEL, 2, 4, 8)
alt_6 = special_mlp(D_MODEL, 6, 4, 8)
alt_two = alt_2 + alt_6
alt_one = special_mlp(D_MODEL, 8, 4, 8)

print()
print(T.table(["구성", "저장소 코드대로 센 값", "모델 대비", "논문 표 6", "논문 %"],
              [["MLP×2 (two_mlp, 본 모델)", f"{code_two:,} ({T.fmt_params(code_two)})",
                f"{code_two / model_total * 100:.3f}%", "3.7M", "0.11%"],
               ["MLP×1 (mlp, 출력 8개)", f"{code_one:,} ({T.fmt_params(code_one)})",
                f"{code_one / model_total * 100:.3f}%", "1.8M", "0.05%"],
               ["(SSM+MLP)×2", f"{2 * blk_total + code_two:,} "
                f"({T.fmt_params(2 * blk_total + code_two)})",
                f"{(2 * blk_total + code_two) / model_total * 100:.3f}%",
                "45.2M", "1.3%"]], highlight={0}))

print(f"""
→ **논문과 코드가 맞지 않는다.** two_mlp를 저장소 코드대로 세면
   {T.fmt_params(code_two)}인데 논문 표 6은 3.7M이라 적는다. {code_two / 3_700_000:.2f}배 차이.

가장 그럴듯한 설명 (가설, 미확인)
  SpecialMLP의 축소비를 //2, //4가 아니라 **//4, //8**로 두면
    출력 2개 → {alt_2:,} / 출력 6개 → {alt_6:,} / 합계 {alt_two:,} = {alt_two / 1e6:.2f}M ≈ 3.7M ✓
    출력 8개 하나만 → {alt_one:,} = {alt_one / 1e6:.2f}M ≈ 1.8M ✓
  즉 논문 표 6은 축소비가 한 단계 더 큰(=더 작은) 헤드로 잰 값이고,
  공개된 코드는 그보다 넓은 헤드다. 논문 제출 후 코드를 바꿨을 가능성.
  (SSM+MLP)×2의 45.2M도 'Mamba 블록 1개({T.fmt_params(blk_total)}) + 위 3.7M'
  = {(blk_total + alt_two) / 1e6:.1f}M에 가깝지만, 코드는 ssm1·ssm2 **두 개**를
  만든다(manip.py 56~57행) → {T.fmt_params(2 * blk_total + code_two)}. 여기도 상충.

이 불일치가 논문의 결론을 흔드는가? — 아니다. 오히려 강화한다.
  · {T.fmt_params(code_two)}든 3.7M이든 전체의 0.1~0.3%다. "10배 작다"는
    ManipLLM 41.3M(0.5%) 대비 주장은 두 값 모두에서 성립한다.
  · 표 6의 결론 자체가 "헤드 크기는 결과에 거의 영향이 없다"
    (63.7% / 62.1% / 63.2%)이므로 크기가 두 배여도 논지가 유지된다.
  · 다만 **논문 숫자를 그대로 인용해 재현하려 하면 맞지 않는다.**
    대외 문서에 쓸 때는 "논문 표 6 기준 3.7M(공개 코드로는 {T.fmt_params(code_two)},
    확인 2026-09-11)"처럼 두 값을 같이 적는 편이 안전하다.""")

# ─────────────────────────────────────────────────────────────────────────────
# (4) 베이스라인과 비교 — 논문 4.3절
# ─────────────────────────────────────────────────────────────────────────────
print("""
(4) "10배 작다"는 무엇과 비교한 것인가 (논문 4.3절)
────────────────────────────────────────────────────────────────""")
print(T.table(["모델", "파인튜닝하는 것", "파라미터", "모델 대비"],
              [["RoboFlamingo", "모델 일부 전체", "1.8B", "35.5%"],
               ["ManipLLM", "어댑터", "41.3M", "0.5%"],
               ["OpenVLA", "모델 전체 (그림 1)", "7.0B", "100%"],
               ["RoboMamba (논문)", "MLP 헤드 2개", "3.7M", "0.11%"],
               ["RoboMamba (공개 코드)", "MLP 헤드 2개",
                T.fmt_params(code_two), f"{code_two / model_total * 100:.2f}%"]],
              highlight={3, 4}))
print(f"""
비교의 성격을 정확히 보기
  · RoboFlamingo·ManipLLM은 백본까지 손대므로 원래 능력이 훼손될 수 있다
    (논문 3.4절: "breaks the inherent abilities of the pre-trained model").
    RoboMamba는 백본을 아예 동결하니 언어 추론 능력이 그대로 남는다.
    이것이 FiS-VLA가 L_slow(공동학습)로 해결하려 한 문제와 같은 문제인데,
    RoboMamba는 '건드리지 않음'으로, FiS-VLA는 '같이 학습함'으로 풀었다.
  · 단, 동결의 대가도 있다. 조작 학습이 백본의 표현을 개선할 수 없으므로
    백본이 못 담은 정보는 헤드가 만들어 낼 수 없다(실습 5 (3)).
  · 저장 용량으로는 헤드가 fp32에서 {code_two * 4 / 1e6:.1f} MB
    (논문은 "only a 7MB policy head" — 3.7M × 4바이트 ≈ 14.8MB가 아니라
     7MB라면 3.7M × 2바이트(fp16) 또는 1.8M × 4바이트에 해당. 미확인).

해 볼 것
  · D_MODEL을 1.4b 설정(d_model=2048, n_layer=48)으로 바꿔 전체를 다시 세 보라.
    논문 그림 3 a)의 'Ours-1.4B'가 몇 개짜리 모델인가?
  · in_proj가 블록의 64%를 먹는다. expand를 2에서 1로 줄이면 전체가 얼마나
    줄고, 실습 1의 상태 크기는 어떻게 되는가?""")


---

## 실습 7 · 논문 문장을 저장소 줄 번호로

```
논문의 주장 하나하나가 코드의 어느 줄인지 직접 찾아 보여 준다.
줄 번호는 바뀌므로 **앵커 문자열**로 찾는다. 앵커가 사라지면 '찾지 못함'을
출력하고 계속 진행한다.

처음 실행하면 GitHub에서 파일 6개(약 200KB)를 받아 out/src/에 저장한다.
두 번째부터는 --offline로 네트워크 없이 돌린다.

실행: python3 lab7_code_map.py            (처음, 약 5초)
      python3 lab7_code_map.py --offline   (이후)
```

원본 스크립트: `lab7_code_map.py`


In [ ]:
"""실습 7 · 논문 문장을 저장소 줄 번호로 — 가이드 2.4 · 2.5 · 3.4

논문의 주장 하나하나가 코드의 어느 줄인지 직접 찾아 보여 준다.
줄 번호는 바뀌므로 **앵커 문자열**로 찾는다. 앵커가 사라지면 '찾지 못함'을
출력하고 계속 진행한다.

처음 실행하면 GitHub에서 파일 6개(약 200KB)를 받아 out/src/에 저장한다.
두 번째부터는 --offline로 네트워크 없이 돌린다.

실행: python3 lab7_code_map.py            (처음, 약 5초)
      python3 lab7_code_map.py --offline   (이후)
"""

import os
import sys
import urllib.request

import tinyssm as T

HERE = "."
SRC = os.path.join(HERE, "out", "src")
BASE = "https://raw.githubusercontent.com/lmzpai/roboMamba/main"
FILES = ["readme.md", "requirements.txt", "src/script/test.sh", "src/model/vision.py",
         "src/model/vlm.py", "src/model/manip.py", "src/model/llm.py",
         "src/model/create.py"]
OFFLINE = os.path.exists(os.path.join(SRC, "readme.md"))

# (파일, 앵커, 논문에서의 위치, 무엇을 확인하는가)
ANCHORS = [
    ("src/model/vision.py", "vision_encoders = {", "표 1 'Res.' 열 · 부록 표 4",
     "고를 수 있는 비전 인코더 네 개. test.sh는 CLIP224를 쓴다"),
    ("src/model/vision.py", "get_intermediate_layers", "3.2절 (논문에 없는 사실)",
     "마지막 층이 아니라 끝에서 두 번째 층을 꺼낸다"),
    ("src/model/llm.py", "mamba_dict = {", "4.1절 구현 상세 · 그림 3 a)",
     "mamba-2.8b / 1.4b / 790m / 370m. 논문 표는 2.7B, 그림 3은 1.4B도 비교"),
    ("src/model/vlm.py", "class Projector", "3.2절 'simple cross-modal connector'",
     "논문은 MLP라고만 쓴 연결기의 실제 모양 (3층, 중간 4배 확장)"),
    ("src/model/vlm.py", "vision_encoded_result, text_encoded", "3.2절 cat(f_v^L, f_t)",
     "이어 붙이는 순서가 [이미지, 텍스트]. Mamba라 순서가 결과를 바꾼다(실습 4)"),
    ("src/model/vlm.py", "target_modules=", "3.3절 (논문에 없는 사실)",
     "LoRA를 쓸 때 건드리는 Mamba 모듈 네 개"),
    ("src/model/manip.py", "class SpecialMLP", "3.4절 'simple policy head' · 부록 표 6",
     "정책 헤드의 실제 층 구성. 실습 6에서 파라미터를 센 대상"),
    ("src/model/manip.py", "self.action_head1 = SpecialMLP", "3.4절 'two types of MLPs'",
     "위치 2개 / 방향 6개로 나뉜 두 헤드 (head_type='two_mlp')"),
    ("src/model/manip.py", "self.llm.mamba.lm_head = nn.Identity()", "3.4절 (논문에 없는 사실)",
     "lm_head를 지워 .logits이 2560차원 은닉상태가 되게 하는 기법"),
    ("src/model/manip.py", "res = (res[:,vision_encoded.shape[1]]", "그림 2 'pooling operation'",
     "논문은 pooling이라 쓰지만 실제로는 토큰 두 개의 평균 → 상충"),
    ("src/model/manip.py", "def bgs(d6s)", "3.4절 a_dir ∈ R^(3×3)",
     "6개 숫자 → Gram-Schmidt → 회전행렬. 논문 본문에 6D 얘기는 없다"),
    ("src/model/manip.py", "theta = torch.clamp(0.5 * (Rt - 1)", "식 (6) 방향 손실",
     "arccos((tr−1)/2)와 정의역 보호 clamp"),
    ("src/model/manip.py", "self.ssm1 = MambaBlock", "부록 표 6 '(SSM block+MLP)×2'",
     "SSM 블록을 두 개 만든다. 45.2M과 맞지 않는 지점(실습 6)"),
    ("src/script/test.sh", "--llm_name mamba-2.8b", "4.1절 구현 상세",
     "평가 스크립트의 기본 설정. run_type VLM = 추론 평가, dataset robovqa"),
    ("readme.md", "The checkpoints are shown in the test branch", "공개 범위",
     "학습 코드는 공개되지 않았다 — 저자에게 메일로 요청해야 한다"),
    ("requirements.txt", "mamba_ssm", "재현 환경",
     "CUDA 커널 패키지. 이것 때문에 Apple Silicon에서는 저장소 원본이 안 돈다"),
]

print(T.header(7, "논문 문장을 저장소 줄 번호로", "2.4 · 2.5 · 3.4"))
print(f"\n저장소: https://github.com/lmzpai/roboMamba  (main, 확인 2026-09-11)")

os.makedirs(SRC, exist_ok=True)
for f in FILES:
    dst = os.path.join(SRC, f)
    if os.path.exists(dst):
        continue
    if OFFLINE:
        print(f"  [offline] 없음: {f}")
        continue
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    try:
        with urllib.request.urlopen(f"{BASE}/{f}", timeout=20) as r:
            data = r.read()
        with open(dst, "wb") as g:
            g.write(data)
        print(f"  받음 {f}  ({len(data):,} B)")
    except Exception as e:
        print(f"  실패 {f}: {e}")

cache = {}
for f in FILES:
    p = os.path.join(SRC, f)
    if os.path.exists(p):
        cache[f] = open(p, encoding="utf-8", errors="replace").read().splitlines()

found = missing = 0
cur_file = None
for f, anchor, where, why in ANCHORS:
    if f != cur_file:
        cur_file = f
        print(f"\n{'━' * 74}\n  {f}\n{'━' * 74}")
    lines = cache.get(f)
    if lines is None:
        print(f"  · 파일 없음 (offline?)  앵커 {anchor!r}")
        missing += 1
        continue
    hit = next((i for i, ln in enumerate(lines)
                if anchor.replace(" ", "") in ln.replace(" ", "")), None)
    if hit is None:
        print(f"  · 찾지 못함: {anchor!r}  ← 저장소가 바뀌었을 수 있다")
        missing += 1
        continue
    found += 1
    print(f"\n  ▸ {why}")
    print(f"    논문: {where}")
    lo, hi = max(0, hit - 1), min(len(lines), hit + 4)
    for i in range(lo, hi):
        mark = "▸" if i == hit else " "
        print(f"    {mark} {i + 1:4d} │ {lines[i][:96]}")

print(f"""
{'━' * 74}
앵커 {found}개 확인, {missing}개 실패.

이 표를 들고 확인할 것 (스터디 질문 후보)
  1. 논문 그림 2는 '전역 토큰을 pooling으로 만든다'고 하는데 코드는 토큰
     두 개의 평균이다. 어느 쪽이 표 2의 63%를 낸 설정인가?
  2. 부록 표 6의 3.7M과 코드의 8.20M 중 어느 쪽이 체크포인트와 맞는가?
     (test 브랜치 체크포인트를 열어 헤드 텐서 모양을 보면 판정된다)
  3. 학습 코드가 없으므로 Stage 1.1 → 1.2 → 2 파이프라인은 논문 서술만으로
     재현해야 한다. 협업 논의라면 여기가 가장 먼저 물어볼 지점이다.

저장소에 **없는** 것 (확인 2026-09-11)
  · 학습 코드 (readme: 메일로 요청)
  · SAPIEN 데이터 수집 스크립트, RoboVQA 전처리
  · 조작 평가 스크립트 (test.sh는 run_type VLM = 추론 평가만)
  · 체크포인트 (main 아님, test 브랜치)
  → FiS-VLA 저장소(학습·평가 스크립트 모두 공개)와 공개 범위가 크게 다르다.
    같은 회사가 공저자인데 왜 다른지도 부스 질문 후보다.""")


---

## 부록 · 진짜 Mamba 모델을 돌려 보기 (선택)

실습 1~7은 numpy 축소판이다. 여기서는 **실제 사전학습된 Mamba LLM**을 불러 RoboMamba의 백본이 무엇인지 눈으로 본다. RoboMamba 자체의 체크포인트는 저장소 `test` 브랜치에 있고 학습 코드는 비공개이므로, 백본인 `state-spaces/mamba-*-hf`까지만 확인한다.

| 환경 | 설치 | 130m 생성 속도 (측정값) |
|---|---|---|
| 코랩 (T4 GPU) | 기본 torch 사용 | 빠름 |
| Mac (Apple Silicon, MPS) | `pip install torch transformers` | **89~145 tok/s** |
| Mac (CPU) | 같음 | 0.83 tok/s — 쓰지 말 것 |

`mamba_ssm` / `causal_conv1d`(저장소 requirements.txt)는 **CUDA 전용**이라 Mac에서는 빌드되지 않는다. 하지만 transformers가 순수 PyTorch 대체 경로를 갖고 있어("falling back to its reference PyTorch implementation... This is correct but much slower") 결과는 같다. 측정: M2 Pro / torch 2.9.1 / transformers 5.1.1, 2026-09-11.


In [ ]:
# 필요할 때만 실행 (약 1.5GB 다운로드). 실습 1~7은 이것 없이도 전부 동작한다.
# %pip install -q torch transformers

import time

try:
    import torch
    from transformers import AutoTokenizer, MambaForCausalLM
    HAVE_TORCH = True
except ImportError:
    HAVE_TORCH = False
    print("torch/transformers 가 없다. 위 %pip 줄의 주석을 풀고 이 셀을 다시 실행하라.")
    print("이 셀은 선택 사항이다 — 실습 1~7은 numpy만으로 모두 동작한다.")


def run_real_mamba(name="state-spaces/mamba-130m-hf"):
    if torch.cuda.is_available():
        dev = "cuda"                  # 코랩 T4
    elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        dev = "mps"                   # Apple Silicon — CPU보다 100배 빠르다
    else:
        dev = "cpu"                   # 0.83 tok/s. 권하지 않는다
    print("장치:", dev)

    tok = AutoTokenizer.from_pretrained(name)
    m = MambaForCausalLM.from_pretrained(name).to(dev).eval()
    c = m.config
    print(f"{name}: {sum(p.numel() for p in m.parameters()) / 1e6:.1f}M 파라미터")
    print(f"  hidden={c.hidden_size}  layers={c.num_hidden_layers}  "
          f"d_state={c.state_size}  d_conv={c.conv_kernel}  "
          f"expand={c.expand}  dt_rank={c.time_step_rank}")
    print("  → RoboMamba가 쓰는 2.8b는 hidden=2560, layers=64 (실습 6에서 센 값)")

    # 실습 1에서 '저장소 초기값은 A = −(1..16)'이라 했다. 학습 후에는 어떻게 됐나?
    A = -torch.exp(m.backbone.layers[0].mixer.A_log.float()).detach().cpu()
    print(f"\n블록 0의 A (학습 후): 모양 {tuple(A.shape)}")
    print(f"  최솟값 {A.min():.2f}   최댓값 {A.max():.2f}")
    print(f"  채널 0의 {A.shape[1]}개 값: {[round(v, 2) for v in A[0].tolist()]}")
    print("  → S4D 초기값 −(1..16)에서 크게 벗어나 있다. 절댓값이 작은 채널")
    print("     (−0.1 근처)은 수백 토큰을 기억하고, 큰 채널(−200 이하)은 거의")
    print("     직전 토큰만 본다. 실습 1 (3)의 '기억 반경'이 채널마다 학습으로")
    print("     분화한 결과다 — 사람이 정해 준 것이 아니다.")

    ids = tok("The robot should first", return_tensors="pt").input_ids.to(dev)
    with torch.no_grad():
        m.generate(ids, max_new_tokens=3, do_sample=False)          # 워밍업
        t0 = time.time()
        out = m.generate(ids, max_new_tokens=30, do_sample=False)
    dt = time.time() - t0
    print(f"\n생성 30토큰 {dt:.2f}초 ({30 / dt:.1f} tok/s)")
    print("→", tok.decode(out[0], skip_special_tokens=True))
    print("\n(130m은 로봇 데이터를 본 적이 없다. 답이 엉성한 것이 정상이고,")
    print(" 논문 Stage 1.2 instruction co-training이 무엇을 하는지 역으로 보여 준다.)")


if HAVE_TORCH:
    run_real_mamba()


---

## 다음 단계

- 실습이 가리키는 논문 절은 **RoboMamba 정독 가이드**를 함께 보라
- 저장소 원본(`bash script/test.sh`)을 돌리려면 CUDA GPU + `test` 브랜치 체크포인트가 필요하다. 학습 코드는 비공개(저자 메일 요청)
- 같은 스터디 1주차의 다른 논문 **FiS-VLA**(arXiv 2506.01953)는 별도 실습 폴더가 있다
